# Experimento 03: Final SAM con red de nombres

**Estado:** rama experimental conservada para trazabilidad, no estrategia final de entrega.

**Deriva de:** 05_nn_sam_entrenamiento_completo -> clasificador anatomico adicional.

**Por que se conserva:** Se conserva porque mostro que segmentar y nombrar no eran el mismo problema.

Este notebook se deja con salidas limpias para que GitHub sea liviano. La estrategia principal queda en `notebooks/`; esta carpeta explica los caminos que se probaron y por que no todos terminaron como modelo final.

<!-- codex-experimento-conservado -->


# Final SAM - NN-SAM + MedSAM optimizado

Este notebook conserva solamente la estrategia ganadora encontrada durante las pruebas previas. Se eliminan rutas diagnosticas y comparativos que ya no se usaran en la version final.

Estrategia final:

1. Entrenar una red ligera NN-SAM para generar cajas vertebrales automaticas.
2. Usar esas cajas como prompt `box_only` para MedSAM.
3. Entrenar MedSAM en dos pasos: primero `mask_decoder`, luego `mask_decoder + ultimo bloque del encoder`.
4. Evaluar la ruta final `medsam_decoder_encoder_parcial + box_only + sin_pad`.

La conclusion previa fue que la segmentacion espacial generaliza bien. El punto a optimizar en adelante es la asignacion anatomica automatica, especialmente en escoliosis y GT parcial.


## 1. Configuracion final

Esta celda contiene solo los parametros necesarios para la ruta final. La carpeta de resultados se separa de las corridas exploratorias para poder optimizar esta version sin mezclar archivos.

Se conserva CUDA obligatorio porque se entrena NN-SAM y se ajusta MedSAM. La unica expansion de caja que queda es `sin_pad`, porque en validacion fue superior a las cajas ampliadas.


In [ ]:
# --- Librerias y reproducibilidad ---
import json
import math
import sys
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- GPU: esta corrida debe usar CUDA para evitar tiempos impracticos ---
REQUIERE_CUDA = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIERE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError("CUDA no esta disponible. Esta prueba esta pensada para GPU.")

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print("GPU:", torch.cuda.get_device_name(0))

# --- Rutas de datos y carpeta de resultados ---
DATASET_ROOT = Path("C:/Users/luisf/Downloads/ProyectoFinal/Scoliosis_Dataset")
MEDSAM_DATA_ROOT = Path("C:/Users/luisf/Downloads/ProyectoFinal/dataset_procesado_scoliosis_medsam/medsam")
LABELS_DICT_PATH = DATASET_ROOT / "diccionario_etiquetas_T1_T12_L1_L5.json"
RESULTADOS_DIR = Path("C:/Users/luisf/Downloads/ProyectoFinal/resultados_final_sam")
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

# --- Hiperparametros de la red de cajas NN-SAM ---
IMG_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 140
PACIENCIA = 22
LR = 6e-4
BASE_CH = 32

SIGMA_CENTRO = 4.5
BOX_EXPAND_W = 1.12
BOX_EXPAND_H = 1.12
PRESENCE_THR = 0.35
USAR_PRESENCE_EN_DECODIFICACION = False
MIN_VISIBLE_LABELS = 8
MAX_CANDIDATOS = 120
TOP_PICOS = 90
MIN_DIST_PICOS = 8
Y_MIN_ANATOMICO_MARGEN = 0.06
CRANEO_SCORE_FACTOR = 0.12
MAX_GAP_REL_DY = 2.40

# --- Aumentacion y evaluacion de cajas ---
# Entrenamiento final de cajas: cada epoca re-muestrea augmentaciones distintas.
AUGMENT_REPEATS = 6
NUM_WORKERS = 0  # En Windows/Jupyter es mas estable; subirlo solo si el entorno lo permite.
TARGET_BOX_EXPAND_W = 1.10
TARGET_BOX_EXPAND_H = 1.12

# Evaluacion principal. Val completo mide comportamiento general; los paneles visuales se limitan a casos trazadores.
EVALUAR_TODO_VAL = True
EVALUAR_TEST_FINAL = True
PACIENTES_VISUALIZACION = ["N_12", "N_32", "S_187", "S_130", "S_190", "S_80"]

# --- Entrenamiento y prueba de MedSAM ---
# MedSAM usa cajas automaticas y permite una expansion leve adicional del prompt.
MEDSAM_EXPANSIONES_BBOX = [
    {"nombre": "sin_pad", "frac_x": 0.00, "frac_y": 0.00},
]
MEDSAM_GUARDAR_FIGURAS = True

# Entrenamiento MedSAM dentro de este mismo notebook final.
# Se entrena decoder como paso intermedio y luego encoder parcial como modelo final.
EJECUTAR_ENTRENAMIENTO_MEDSAM = True
MEDSAM_TRAIN_BATCH_SIZE = 2
MEDSAM_TRAIN_WORKERS = 0
MEDSAM_TRAIN_FRAC_X = 0.04
MEDSAM_TRAIN_FRAC_Y = 0.06
MEDSAM_DECODER_EPOCHS = 12
MEDSAM_DECODER_PATIENCIA = 4
MEDSAM_ENCODER_PARCIAL_EPOCHS = 8
MEDSAM_ENCODER_PARCIAL_PATIENCIA = 3
MEDSAM_LR_DECODER = 1e-4
MEDSAM_LR_DECODER_REFINO = 5e-5
MEDSAM_LR_ENCODER = 1e-5
MEDSAM_WEIGHT_DECAY = 1e-4
MEDSAM_PESO_ESCOLIOSIS = 1.25
MEDSAM_PESO_NORMAL = 1.00
MEDSAM_VERTEBRAS_DIFICILES = ["T1", "T2", "T3", "L4", "L5"]
MEDSAM_PESO_VERTEBRA_DIFICIL = 1.15

# --- Red neuronal de nombres anatomicos ---
# Esta capa intenta convertir buenas cajas/segmentaciones flexibles en mejores etiquetas estrictas.
USAR_RED_NOMBRES_ANATOMICOS = True
EJECUTAR_ENTRENAMIENTO_NOMBRES = True
NOMBRES_EPOCHS = 120
NOMBRES_PACIENCIA = 18
NOMBRES_LR = 8e-4
NOMBRES_WEIGHT_DECAY = 1e-4
NOMBRES_MIN_IOU_TARGET = 0.12
NOMBRES_MIN_RECALL_TARGET = 0.45
NOMBRES_EXTRA_LABEL = "__extra__"
NOMBRES_EXTRA_ID = 17
NOMBRES_SKIP_PENALTY = 0.08
NOMBRES_EXTRA_PENALTY = 0.03
NOMBRES_MIN_NOMBRADAS_FALLBACK = 8
NOMBRES_CKPT_PATH = RESULTADOS_DIR / "red_nombres_anatomicos_best.pt"
NOMBRES_REGENERAR_DATASET = True
NOMBRES_BATCH_SIZE = 8
NOMBRES_HIDDEN = 96
NOMBRES_MIN_PROB_LABEL = 0.05
NOMBRES_USAR_DP_MONOTONICA = True

# --- Decodificacion anatomica de la columna ---
# Ruta principal: 17 centros y etiquetado desde T1.
# La prueba con L5 como ancla queda solo como diagnostico porque desplazo casos buenos.
N_CAJAS_CAMINO = 17
ETIQUETADO_MODO = "top_anchor"  # top_anchor | bottom_anchor | auto_anchor
ESTRATEGIAS_ETIQUETADO = ["top_anchor"]
AUTO_BOTTOM_Y_REL = 0.82
AUTO_BOTTOM_MIN_EXTRA_BOXES = 4

LAMBDA_HEAT = 1.00
LAMBDA_WH = 0.40
LAMBDA_OFF = 0.12
LAMBDA_PRESENCE = 0.00

CKPT_PATH = RESULTADOS_DIR / "cajas_nn_sam_entrenamiento_completo_best.pt"

print("Device:", DEVICE)
print("IMG_SIZE:", IMG_SIZE)
print("Resultados:", RESULTADOS_DIR)


# Esta version entrena la red de cajas y luego separa evaluacion de prompts de segmentacion MedSAM.
NN_SAM_DIR = RESULTADOS_DIR
EJECUTAR_MEDSAM_BASICO = False
MEDSAM_REPO_DIR = Path("C:/Users/luisf/MedSAM")
if MEDSAM_REPO_DIR.exists() and str(MEDSAM_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(MEDSAM_REPO_DIR))

MEDSAM_CKPT_PATH = Path("C:/Users/luisf/MedSAM/work_dir/MedSAM/medsam_vit_b.pth")
MEDSAM_FINETUNED_PATH = Path("C:/Users/luisf/Downloads/ProyectoFinal/best_medsam_spine_maskdecoder.pth")
MEDSAM_DEMO_PATIENT_ID = "S_187"
MEDSAM_DEMO_SPLIT = "val"
MEDSAM_DEMO_USAR_SHIFT = True
MEDSAM_DEMO_MAX_VERTEBRAS = None  # None segmenta todas las vertebras detectadas del caso demo.



# --- Evaluacion final MedSAM ---
# Evaluacion final MedSAM: solo la estrategia ganadora con prompts automaticos NN-SAM.
EJECUTAR_COMPARATIVO_MEDSAM = True
MEDSAM_COMPARAR_PATIENT_IDS = ["N_12", "N_32", "S_187", "S_130", "S_190", "S_80"]  # panel corto de validacion visual
MEDSAM_COMPARAR_USAR_SHIFT = True
MEDSAM_COMPARAR_PROMPT_MODE = "box_only"  # box_only fue la ruta que dejo de fragmentar las mascaras.
MEDSAM_DECODER_PATH = RESULTADOS_DIR / "medsam_decoder_entrenado_nn_sam.pt"
MEDSAM_ENCODER_DECODER_PATH = RESULTADOS_DIR / "medsam_decoder_encoder_parcial_entrenado_nn_sam.pt"


## 2. Carga de datos

Se leen los splits ya definidos `train`, `val` y `test`. Esta version no vuelve a partir el dataset: usa las carpetas existentes del dataset procesado.

Tambien se auditan las etiquetas realmente presentes en cada mascara. Esto se conserva porque en varias imagenes el GT es parcial; por tanto no se debe exigir que todas tengan las 17 vertebras anotadas.


In [ ]:
# Lee el diccionario oficial de etiquetas T1-L5.
with open(LABELS_DICT_PATH, "r", encoding="utf-8") as f:
    LABELS_DICT = json.load(f)

MAPEO_ID = {int(k): v for k, v in LABELS_DICT["mascara_multiclase_id_png"].items()}
CLASES_OBJETIVO = [MAPEO_ID[i] for i in range(1, 18)]
N_CLASES = len(CLASES_OBJETIVO)
VERTEBRA_TO_ID = {nombre: idx for idx, nombre in MAPEO_ID.items() if idx != 0}
ID_TO_VERTEBRA = {idx: nombre for nombre, idx in VERTEBRA_TO_ID.items()}

SPLITS = ["train", "val", "test"]
EXTS_IMG = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]


def buscar_archivo_por_stem(carpeta, stem):
    for ext in EXTS_IMG:
        path = carpeta / f"{stem}{ext}"
        if path.exists():
            return path
    return None


def resolver_paths_split(split):
    split_dir = MEDSAM_DATA_ROOT / split
    prompts_path = split_dir / "prompts.json"
    image_dir = next((split_dir / d for d in ["images", "imgs", "image"] if (split_dir / d).is_dir()), None)
    mask_dir = next((split_dir / d for d in ["masks", "labels", "mask", "annotations"] if (split_dir / d).is_dir()), None)
    if image_dir is None or mask_dir is None or not prompts_path.exists():
        raise FileNotFoundError(f"Split incompleto: {split_dir}")
    return {"image_dir": image_dir, "mask_dir": mask_dir, "prompts_path": prompts_path}


def cargar_imagen(path):
    return np.array(Image.open(path).convert("RGB"))


def cargar_mascara(path):
    return np.array(Image.open(path)).astype(np.int32)


# Convierte prompts.json a un diccionario patient_id -> vertebra -> bbox.
def normalizar_prompts(prompts_split):
    out = {}
    for item in prompts_split:
        patient_id = str(item.get("patient_id"))
        sub = {}
        prompts_item = item.get("prompts", {})
        iterable = prompts_item.values() if isinstance(prompts_item, dict) else prompts_item
        for info in iterable:
            if not isinstance(info, dict):
                continue
            vertebra = str(info.get("vertebra", "")).upper()
            if vertebra in CLASES_OBJETIVO and "bbox_xyxy" in info:
                sub[vertebra] = dict(info)
        if sub:
            out[patient_id] = sub
    return out


def resolver_paths_muestra(split, patient_id):
    info = SPLIT_INFO[split]
    path_img = buscar_archivo_por_stem(info["image_dir"], patient_id)
    path_mask = buscar_archivo_por_stem(info["mask_dir"], patient_id)
    if path_img is None or path_mask is None:
        raise FileNotFoundError(f"No encontre imagen/mascara para {patient_id}")
    return path_img, path_mask


def vertebras_presentes_en_mascara(mask):
    ids = sorted(int(v) for v in np.unique(mask) if 1 <= int(v) <= N_CLASES)
    return [ID_TO_VERTEBRA[i] for i in ids]


def filtrar_prompts_por_gt(prompts_sample, labels_gt):
    permitidas = set(labels_gt)
    return {v: info for v, info in prompts_sample.items() if v in permitidas}


# Audita que vertebras existen realmente en la mascara de cada imagen.
def construir_gt_labels_dicc():
    dicc = {split: {} for split in SPLITS}
    filas = []
    for split in SPLITS:
        for patient_id in sorted(PROMPTS_DICC[split].keys()):
            _, path_mask = resolver_paths_muestra(split, patient_id)
            labels_gt = vertebras_presentes_en_mascara(cargar_mascara(path_mask))
            labels_prompt = sorted(PROMPTS_DICC[split][patient_id].keys(), key=lambda v: VERTEBRA_TO_ID[v])
            extras_prompt = [v for v in labels_prompt if v not in labels_gt]
            faltantes_prompt = [v for v in labels_gt if v not in labels_prompt]
            dicc[split][patient_id] = labels_gt
            filas.append({
                "split": split,
                "patient_id": patient_id,
                "n_gt": len(labels_gt),
                "n_prompts_json": len(labels_prompt),
                "n_prompts_extra_vs_gt": len(extras_prompt),
                "n_gt_sin_prompt_json": len(faltantes_prompt),
                "labels_gt": ",".join(labels_gt),
                "labels_prompt_extra_vs_gt": ",".join(extras_prompt),
                "labels_gt_sin_prompt_json": ",".join(faltantes_prompt),
            })
    return dicc, pd.DataFrame(filas)


SPLIT_INFO = {split: resolver_paths_split(split) for split in SPLITS}
PROMPTS = {}
for split in SPLITS:
    with open(SPLIT_INFO[split]["prompts_path"], "r", encoding="utf-8") as f:
        PROMPTS[split] = json.load(f)

PROMPTS_DICC = {split: normalizar_prompts(PROMPTS[split]) for split in SPLITS}
GT_LABELS_DICC, DF_AUDITORIA_GT = construir_gt_labels_dicc()

for split in SPLITS:
    print(f"{split}: {len(PROMPTS_DICC[split])} pacientes")
print("Clases:", CLASES_OBJETIVO)
display(
    DF_AUDITORIA_GT.groupby("split", as_index=False)
    .agg(
        pacientes=("patient_id", "count"),
        n_gt_promedio=("n_gt", "mean"),
        prompts_extra_vs_gt=("n_prompts_extra_vs_gt", "sum"),
        gt_sin_prompt_json=("n_gt_sin_prompt_json", "sum"),
    )
)


## 3. Plantilla de tamano

Se conserva la plantilla de tamano mediana desde `train` como respaldo para la red de cajas. No es una estrategia alternativa; solo evita predicciones absurdas si algun ancho/alto sale inestable.


In [ ]:
def construir_template_bbox_train(prompts_dicc, split_template="train"):
    filas = []
    for patient_id, prompts_sample in prompts_dicc[split_template].items():
        for vertebra, info in prompts_sample.items():
            x0, y0, x1, y1 = info["bbox_xyxy"]
            filas.append({
                "patient_id": patient_id,
                "vertebra": vertebra,
                "id_real": VERTEBRA_TO_ID[vertebra],
                "cx_rel": ((x0 + x1) / 2) / 1024,
                "cy_rel": ((y0 + y1) / 2) / 1024,
                "w_rel": (x1 - x0) / 1024,
                "h_rel": (y1 - y0) / 1024,
            })

    df = pd.DataFrame(filas)
    template = (
        df.groupby(["vertebra", "id_real"], as_index=False)
        .agg(cx_rel=("cx_rel", "median"), cy_rel=("cy_rel", "median"), w_rel=("w_rel", "median"), h_rel=("h_rel", "median"))
        .sort_values("id_real")
        .reset_index(drop=True)
    )
    return template, df


template_bbox, df_template = construir_template_bbox_train(PROMPTS_DICC)
display(template_bbox)


## 4. Targets para la red de cajas NN-SAM

La red aprende centros vertebrales, ancho/alto y offset. Esta etapa es necesaria porque MedSAM necesita prompts automaticos, no cajas manuales o derivadas del GT.

Se mantiene la expansion ligera en los targets de entrenamiento de cajas porque ayudo a que los prompts fueran mas utiles para MedSAM.


In [ ]:
# Expande cajas en el espacio original 1024x1024 antes de crear targets.
def expandir_bbox_1024(bbox, factor_w=1.0, factor_h=1.0):
    x0, y0, x1, y1 = [float(v) for v in bbox]
    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2
    bw = max(1.0, (x1 - x0) * float(factor_w))
    bh = max(1.0, (y1 - y0) * float(factor_h))
    return [
        float(np.clip(cx - bw / 2, 0, 1023)),
        float(np.clip(cy - bh / 2, 0, 1023)),
        float(np.clip(cx + bw / 2, 0, 1023)),
        float(np.clip(cy + bh / 2, 0, 1023)),
    ]


def transformar_prompts(prompts_sample, crop_y0=0, crop_y1=1024, flip=False):
    crop_h = max(crop_y1 - crop_y0, 1)
    scale_y = 1024 / crop_h
    out = {}

    for vertebra, info in prompts_sample.items():
        x0, y0, x1, y1 = [float(v) for v in info["bbox_xyxy"]]

        if flip:
            x0, x1 = 1023 - x1, 1023 - x0

        y0 = (y0 - crop_y0) * scale_y
        y1 = (y1 - crop_y0) * scale_y
        cy = (y0 + y1) / 2

        if cy < 0 or cy > 1023:
            continue

        x0 = float(np.clip(x0, 0, 1023))
        x1 = float(np.clip(x1, 0, 1023))
        y0 = float(np.clip(y0, 0, 1023))
        y1 = float(np.clip(y1, 0, 1023))

        if x1 - x0 < 4 or y1 - y0 < 4:
            continue

        nuevo = dict(info)
        nuevo["bbox_xyxy"] = [x0, y0, x1, y1]
        out[vertebra] = nuevo

    return out


def elegir_crop_vertical(prompts_sample, p_crop=0.40):
    if random.random() > p_crop:
        return 0, 1024

    centros = []
    for info in prompts_sample.values():
        x0, y0, x1, y1 = info["bbox_xyxy"]
        centros.append((y0 + y1) / 2)
    if len(centros) < 5:
        return 0, 1024

    for _ in range(20):
        crop_h = random.randint(680, 1024)
        y0 = random.randint(0, 1024 - crop_h)
        y1 = y0 + crop_h
        n_visible = sum(y0 <= c <= y1 for c in centros)
        if n_visible >= 5:
            return y0, y1

    return 0, 1024


# Carga imagen, aplica augmentacion y filtra prompts a etiquetas realmente disponibles.
def cargar_imagen_entrenamiento(split, patient_id, augment=False):
    path_img, _ = resolver_paths_muestra(split, patient_id)
    img = Image.open(path_img).convert("L")
    prompts = PROMPTS_DICC[split][patient_id]
    labels_gt = GT_LABELS_DICC.get(split, {}).get(patient_id, list(prompts.keys()))
    prompts_gt = filtrar_prompts_por_gt(prompts, labels_gt)
    if prompts_gt:
        prompts = prompts_gt

    crop_y0, crop_y1 = elegir_crop_vertical(prompts) if augment else (0, 1024)
    flip = bool(augment and random.random() < 0.50)

    if crop_y0 != 0 or crop_y1 != 1024:
        img = img.crop((0, crop_y0, 1024, crop_y1)).resize((1024, 1024), Image.BILINEAR)
    if flip:
        img = ImageOps.mirror(img)

    img_np = np.array(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32)
    if augment:
        alpha = np.random.uniform(0.88, 1.12)
        beta = np.random.uniform(-0.05, 0.05) * 255
        ruido = np.random.normal(0, np.random.uniform(0, 4), img_np.shape)
        img_np = np.clip(img_np * alpha + beta + ruido, 0, 255)

    p1, p99 = np.percentile(img_np, [1, 99.5])
    img_np = np.clip((img_np - p1) / (p99 - p1 + 1e-6), 0, 1)
    img_t = torch.from_numpy(img_np[None, ...].astype(np.float32))

    prompts_t = transformar_prompts(prompts, crop_y0=crop_y0, crop_y1=crop_y1, flip=flip)
    return img_t, prompts_t


def draw_gaussian(heat, cx, cy, sigma=SIGMA_CENTRO):
    radius = int(max(2, sigma * 3))
    x0 = max(0, int(cx) - radius)
    x1 = min(IMG_SIZE - 1, int(cx) + radius)
    y0 = max(0, int(cy) - radius)
    y1 = min(IMG_SIZE - 1, int(cy) + radius)
    if x1 <= x0 or y1 <= y0:
        return

    yy, xx = np.mgrid[y0:y1 + 1, x0:x1 + 1]
    g = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2)).astype(np.float32)
    heat[y0:y1 + 1, x0:x1 + 1] = np.maximum(heat[y0:y1 + 1, x0:x1 + 1], g)


# Construye heatmap/wh/offset/presencia para entrenamiento tipo CenterNet.
def targets_desde_prompts(prompts_sample):
    heat = np.zeros((1, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    wh = np.zeros((2, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    off = np.zeros((2, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    mask = np.zeros((1, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    presence = np.zeros((N_CLASES,), dtype=np.float32)

    for vertebra, info in prompts_sample.items():
        if vertebra not in VERTEBRA_TO_ID:
            continue
        x0, y0, x1, y1 = expandir_bbox_1024(info["bbox_xyxy"], TARGET_BOX_EXPAND_W, TARGET_BOX_EXPAND_H)
        cx = ((x0 + x1) / 2) / 1024 * IMG_SIZE
        cy = ((y0 + y1) / 2) / 1024 * IMG_SIZE
        xi = int(np.clip(np.floor(cx), 0, IMG_SIZE - 1))
        yi = int(np.clip(np.floor(cy), 0, IMG_SIZE - 1))

        draw_gaussian(heat[0], cx, cy)
        heat[0, yi, xi] = 1.0
        mask[0, yi, xi] = 1.0
        wh[0, yi, xi] = np.clip((x1 - x0) / 1024, 0.01, 0.40)
        wh[1, yi, xi] = np.clip((y1 - y0) / 1024, 0.01, 0.40)
        off[0, yi, xi] = cx - xi
        off[1, yi, xi] = cy - yi
        presence[VERTEBRA_TO_ID[vertebra] - 1] = 1.0

    return {
        "heat": torch.from_numpy(heat),
        "wh": torch.from_numpy(wh),
        "off": torch.from_numpy(off),
        "mask": torch.from_numpy(mask),
        "presence": torch.from_numpy(presence),
    }


# Dataset de cajas: repetir muestras aumenta augmentaciones por epoca sin duplicar archivos.
class VertebraCenterDataset(Dataset):
    def __init__(self, split, augment=False, patient_ids=None, repeat=1):
        self.split = split
        self.augment = augment
        base_ids = list(patient_ids) if patient_ids is not None else sorted(PROMPTS_DICC[split].keys())
        self.samples = [(split, pid) for pid in base_ids for _ in range(max(1, int(repeat)))]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        split, patient_id = self.samples[idx]
        img_t, prompts_t = cargar_imagen_entrenamiento(split, patient_id, augment=self.augment)
        target = targets_desde_prompts(prompts_t)
        return img_t, target, {"split": split, "patient_id": patient_id}


ds_train = VertebraCenterDataset("train", augment=True, repeat=AUGMENT_REPEATS)
ds_val = VertebraCenterDataset("val", augment=False, repeat=1)
print("Train pacientes:", len(PROMPTS_DICC["train"]), "| muestras/epoca:", len(ds_train), "| Val:", len(ds_val))


## 5. Modelo de cajas

Modelo ligero tipo U-Net/CenterNet. Su funcion en la version final no es segmentar, sino proponer cajas vertebrales automaticas de forma rapida y reproducible.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class CenterNetLite(nn.Module):
    def __init__(self, base=BASE_CH):
        super().__init__()
        self.e1 = ConvBlock(1, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.e4 = ConvBlock(base * 4, base * 6)
        self.b = ConvBlock(base * 6, base * 8)

        self.u4 = ConvBlock(base * 8 + base * 6, base * 6)
        self.u3 = ConvBlock(base * 6 + base * 4, base * 4)
        self.u2 = ConvBlock(base * 4 + base * 2, base * 2)
        self.u1 = ConvBlock(base * 2 + base, base)

        self.heat_head = nn.Conv2d(base, 1, 1)
        self.wh_head = nn.Conv2d(base, 2, 1)
        self.off_head = nn.Conv2d(base, 2, 1)
        self.presence_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(base * 8, N_CLASES),
        )

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        e4 = self.e4(F.max_pool2d(e3, 2))
        b = self.b(F.max_pool2d(e4, 2))

        u4 = F.interpolate(b, size=e4.shape[-2:], mode="bilinear", align_corners=False)
        u4 = self.u4(torch.cat([u4, e4], dim=1))
        u3 = F.interpolate(u4, size=e3.shape[-2:], mode="bilinear", align_corners=False)
        u3 = self.u3(torch.cat([u3, e3], dim=1))
        u2 = F.interpolate(u3, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        u2 = self.u2(torch.cat([u2, e2], dim=1))
        u1 = F.interpolate(u2, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        u1 = self.u1(torch.cat([u1, e1], dim=1))

        return {
            "heat": self.heat_head(u1),
            "wh": self.wh_head(u1),
            "off": self.off_head(u1),
            "presence": self.presence_head(b),
        }


model = CenterNetLite().to(DEVICE)
print("Parametros:", round(sum(p.numel() for p in model.parameters()) / 1e6, 3), "M")


## 6. Entrenamiento de NN-SAM

Se entrena la red de cajas desde cero usando `train` y se selecciona checkpoint con `val_loss`. Esta red genera las cajas que luego usa MedSAM.


In [ ]:
# Focal loss: enfatiza centros verdaderos y reduce falsos positivos del heatmap.
def focal_loss_centernet(logits, target):
    pred = torch.sigmoid(logits).clamp(1e-4, 1 - 1e-4)
    pos = (target >= 0.999).float()
    neg = (target < 0.999).float()
    neg_weights = torch.pow(1 - target, 4)

    pos_loss = -torch.log(pred) * torch.pow(1 - pred, 2) * pos
    neg_loss = -torch.log(1 - pred) * torch.pow(pred, 2) * neg_weights * neg
    num_pos = pos.sum().clamp(min=1.0)
    return (pos_loss.sum() + neg_loss.sum()) / num_pos


def regression_loss_at_centers(pred_logits, target, mask, kind="sigmoid_l1"):
    if kind == "sigmoid_l1":
        pred = torch.sigmoid(pred_logits)
    else:
        pred = pred_logits
    mask2 = mask.expand_as(target)
    denom = mask2.sum().clamp(min=1.0)
    return (torch.abs(pred - target) * mask2).sum() / denom


def loss_batch(outputs, target):
    heat_loss = focal_loss_centernet(outputs["heat"], target["heat"])
    wh_loss = regression_loss_at_centers(outputs["wh"], target["wh"], target["mask"], kind="sigmoid_l1")
    off_loss = regression_loss_at_centers(outputs["off"], target["off"], target["mask"], kind="sigmoid_l1")
    presence_loss = F.binary_cross_entropy_with_logits(outputs["presence"], target["presence"])
    total = (
        LAMBDA_HEAT * heat_loss +
        LAMBDA_WH * wh_loss +
        LAMBDA_OFF * off_loss +
        LAMBDA_PRESENCE * presence_loss
    )
    return total, {
        "heat": float(heat_loss.detach().cpu()),
        "wh": float(wh_loss.detach().cpu()),
        "off": float(off_loss.detach().cpu()),
        "presence": float(presence_loss.detach().cpu()),
    }


def mover_target_device(target):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in target.items()}


def crear_loaders():
    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    val_loader = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader


@torch.no_grad()
def evaluar_loss_loader(loader):
    model.eval()
    losses = []
    partes = []
    for x, target, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        target = mover_target_device(target)
        outputs = model(x)
        loss, parts = loss_batch(outputs, target)
        losses.append(float(loss.detach().cpu()))
        partes.append(parts)
    if not losses:
        return np.nan, {}
    mean_parts = {k: float(np.mean([p[k] for p in partes])) for k in partes[0].keys()}
    return float(np.mean(losses)), mean_parts


# Loop principal de entrenamiento de cajas con early stopping por val_loss.
def entrenar_modelo():
    train_loader, val_loader = crear_loaders()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    best_val = np.inf
    sin_mejora = 0
    hist = []
    t0 = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for x, target, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            x = x.to(DEVICE, non_blocking=True)
            target = mover_target_device(target)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(x)
                loss, parts = loss_batch(outputs, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.detach().cpu()))

        val_loss, val_parts = evaluar_loss_loader(val_loader)
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "val_loss": val_loss,
            "val_heat": val_parts.get("heat", np.nan),
            "val_wh": val_parts.get("wh", np.nan),
            "val_off": val_parts.get("off", np.nan),
            "val_presence": val_parts.get("presence", np.nan),
            "tiempo_min": (time.perf_counter() - t0) / 60,
        }
        hist.append(row)
        print(f"epoch={epoch:03d} train={row['train_loss']:.5f} val={val_loss:.5f} heat={row['val_heat']:.4f} wh={row['val_wh']:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            sin_mejora = 0
            CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
            torch.save({"model": model.state_dict(), "config": {"IMG_SIZE": IMG_SIZE, "N_CLASES": N_CLASES, "BASE_CH": BASE_CH, "TARGET_BOX_EXPAND_W": TARGET_BOX_EXPAND_W, "TARGET_BOX_EXPAND_H": TARGET_BOX_EXPAND_H, "AUGMENT_REPEATS": AUGMENT_REPEATS}}, CKPT_PATH)
        else:
            sin_mejora += 1
            if sin_mejora >= PACIENCIA:
                print("Early stopping.")
                break

    hist = pd.DataFrame(hist)
    hist.to_csv(RESULTADOS_DIR / "historial_entrenamiento_centernet.csv", index=False)
    return hist


# En esta copia si entrenamos por defecto para medir el potencial real de la configuracion final.
ENTRENAR_DESDE_CERO = True

# Al ejecutar desde cero, esta rama genera el checkpoint usado por todo lo siguiente.
if ENTRENAR_DESDE_CERO:
    hist_entrenamiento = entrenar_modelo()
    display(hist_entrenamiento.tail())
else:
    hist_entrenamiento = pd.DataFrame()

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"No existe el checkpoint de cajas: {CKPT_PATH}")

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()
print("Checkpoint de cajas cargado:", CKPT_PATH)


## 7. Decodificacion anatomica de cajas

El heatmap produce candidatos de centro. La ruta vertebral se selecciona con programacion dinamica y se etiqueta de arriba hacia abajo (`top_anchor`), que fue la ruta mas estable en las pruebas previas.


In [ ]:
# Convierte una radiografia RGB a tensor normalizado para inferencia NN-SAM.
def imagen_inferencia_tensor(img_rgb):
    gray = np.array(Image.fromarray(img_rgb).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32)
    p1, p99 = np.percentile(gray, [1, 99.5])
    gray = np.clip((gray - p1) / (p99 - p1 + 1e-6), 0, 1)
    return torch.from_numpy(gray[None, None, ...].astype(np.float32)).to(DEVICE)


@torch.no_grad()
def predecir_outputs(img_rgb):
    model.eval()
    x = imagen_inferencia_tensor(img_rgb)
    out = model(x)
    return {
        "heat": torch.sigmoid(out["heat"])[0, 0].detach().cpu().numpy(),
        "wh": torch.sigmoid(out["wh"])[0].detach().cpu().numpy(),
        "off": torch.sigmoid(out["off"])[0].detach().cpu().numpy(),
        "presence": torch.sigmoid(out["presence"])[0].detach().cpu().numpy(),
    }


# Extrae maximos locales del heatmap como candidatos de centros vertebrales.
def extraer_picos(score_map, n_picos=TOP_PICOS, min_dist=MIN_DIST_PICOS, thr_rel=0.12):
    work = score_map.astype(np.float32).copy()
    picos = []
    max0 = float(work.max())
    if max0 <= 0:
        return picos
    thr = max(max0 * thr_rel, float(np.percentile(work, 90)))

    for _ in range(n_picos):
        idx = int(np.argmax(work))
        y, x = np.unravel_index(idx, work.shape)
        score = float(work[y, x])
        if score < thr:
            break
        picos.append({"x": int(x), "y": int(y), "score": score})
        y0 = max(0, y - min_dist)
        y1 = min(work.shape[0], y + min_dist + 1)
        x0 = max(0, x - min_dist)
        x1 = min(work.shape[1], x + min_dist + 1)
        work[y0:y1, x0:x1] = -np.inf
    return picos


def estimar_y_min_anatomico(img_rgb):
    """Estima un limite superior para penalizar craneo/cuello sin usar GT."""
    gray = np.array(Image.fromarray(img_rgb).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32)
    valores = gray[gray > 0]
    if len(valores) == 0:
        return 0.0, {"motivo": "sin_valores"}

    thr = max(5.0, float(np.percentile(valores, 8)))
    active = (gray > thr).astype(np.float32)
    width = active.mean(axis=1)
    kernel = np.ones(21, dtype=np.float32) / 21
    width_s = np.convolve(width, kernel, mode="same")

    h = IMG_SIZE
    top_med = float(np.median(width_s[: int(0.16 * h)]))
    search0 = int(0.12 * h)
    search1 = int(0.55 * h)
    search = width_s[search0:search1]
    if len(search) == 0:
        return 0.0, {"motivo": "sin_search"}

    mid_p85 = float(np.percentile(search, 85))
    umbral_ensanche = max(0.24, top_med * 1.35)
    if top_med > 0.30 or mid_p85 < umbral_ensanche:
        return 0.0, {"motivo": "sin_craneo_claro", "top_med": top_med, "mid_p85": mid_p85}

    idx = np.where(search > umbral_ensanche)[0]
    if len(idx) == 0:
        return 0.0, {"motivo": "sin_ensanche", "top_med": top_med, "mid_p85": mid_p85}

    y_ensanche = float(search0 + idx[0])
    y_min = max(0.0, y_ensanche - Y_MIN_ANATOMICO_MARGEN * h)
    return y_min, {"motivo": "craneo_probable", "top_med": top_med, "mid_p85": mid_p85, "y_ensanche": y_ensanche}


# Selecciona una ruta vertical coherente de candidatos usando programacion dinamica.
def seleccionar_camino_dp(candidatos, n_pasos=N_CAJAS_CAMINO, max_gap_rel=MAX_GAP_REL_DY):
    if len(candidatos) == 0:
        raise RuntimeError("No hay candidatos de centro.")

    candidatos = sorted(candidatos, key=lambda c: (c["y"], c["x"]))[:MAX_CANDIDATOS]
    n = len(candidatos)
    k = min(int(n_pasos), n)
    if k <= 0:
        raise RuntimeError("No hay suficientes candidatos.")

    xs = np.array([c["x"] for c in candidatos], dtype=np.float32)
    ys = np.array([c["y"] for c in candidatos], dtype=np.float32)
    sc = np.array([c["score"] for c in candidatos], dtype=np.float32)

    template = template_bbox.sort_values("id_real").reset_index(drop=True)
    cy_template = template["cy_rel"].to_numpy(dtype=np.float32) * IMG_SIZE
    dy_template = np.diff(cy_template)
    dy_default = float(np.median(dy_template))

    dp = np.full((k, n), -1e9, dtype=np.float32)
    prev = np.full((k, n), -1, dtype=np.int32)
    dp[0] = sc

    for j in range(1, k):
        expected_dy = float(dy_template[j - 1]) if j - 1 < len(dy_template) else dy_default
        expected_dy = max(expected_dy, 4.0)
        min_gap = max(3.0, expected_dy * 0.35)
        max_gap = max(min_gap + 1.0, expected_dy * max_gap_rel)

        for i in range(n):
            dy = ys[i] - ys[:i]
            valid = (dy >= min_gap) & (dy <= max_gap)
            if not valid.any():
                continue
            dx = np.abs(xs[i] - xs[:i])
            dy_pen = np.abs(dy - expected_dy) / expected_dy
            dx_pen = dx / max(IMG_SIZE * 0.22, 1.0)
            trans = dp[j - 1, :i] - 0.34 * dy_pen - 0.08 * dx_pen
            trans[~valid] = -1e9
            best = int(np.argmax(trans))
            dp[j, i] = sc[i] + trans[best]
            prev[j, i] = best

    end = int(np.argmax(dp[k - 1]))
    if dp[k - 1, end] < -1e8 and max_gap_rel < 8.0:
        return seleccionar_camino_dp(candidatos, n_pasos=n_pasos, max_gap_rel=8.0)

    path = [end]
    for j in range(k - 1, 0, -1):
        end = int(prev[j, end])
        if end < 0:
            break
        path.append(end)
    path = path[::-1]

    if len(path) != k:
        orden = np.argsort(sc)[-k:]
        path = sorted(orden.tolist(), key=lambda i: ys[i])

    return [candidatos[i] for i in path]


def bbox_clip(bbox, H, W):
    x0, y0, x1, y1 = [int(round(float(v))) for v in bbox]
    x0, x1 = np.clip([x0, x1], 0, W - 1)
    y0, y1 = np.clip([y0, y1], 0, H - 1)
    if x1 <= x0:
        x1 = min(W - 1, x0 + 1)
    if y1 <= y0:
        y1 = min(H - 1, y0 + 1)
    return [int(x0), int(y0), int(x1), int(y1)]


def decidir_modo_etiquetado_auto(camino):
    if not camino:
        return "top_anchor", {"motivo": "sin_camino"}
    y_bottom_rel = max(float(c["y"]) for c in camino) / IMG_SIZE
    extra_boxes = max(0, len(camino) - N_CLASES)
    if y_bottom_rel >= AUTO_BOTTOM_Y_REL and extra_boxes >= AUTO_BOTTOM_MIN_EXTRA_BOXES:
        return "bottom_anchor", {"motivo": "region_inferior_visible", "y_bottom_rel": y_bottom_rel, "extra_boxes": extra_boxes}
    return "top_anchor", {"motivo": "sin_ancla_inferior_fuerte", "y_bottom_rel": y_bottom_rel, "extra_boxes": extra_boxes}


def elegir_segmento_y_labels(camino, modo_etiquetado="auto_anchor"):
    modo = modo_etiquetado or ETIQUETADO_MODO
    if modo not in ESTRATEGIAS_ETIQUETADO:
        raise ValueError(f"Estrategia de etiquetado no soportada: {modo}")

    camino = sorted(camino, key=lambda c: (c["y"], c["x"]))
    modo_resuelto = modo
    info = {"modo_solicitado": modo}
    if modo == "auto_anchor":
        modo_resuelto, info_auto = decidir_modo_etiquetado_auto(camino)
        info.update(info_auto)

    n_use = min(N_CLASES, len(camino))
    if modo_resuelto == "bottom_anchor":
        segmento = camino[-n_use:]
        labels = list(range(N_CLASES - n_use, N_CLASES))
    else:
        segmento = camino[:n_use]
        labels = list(range(n_use))

    info["modo_resuelto"] = modo_resuelto
    info["n_camino"] = len(camino)
    info["n_prompts"] = len(segmento)
    return segmento, labels, info


# Convierte centros seleccionados en cajas etiquetadas T1-L5.
def construir_prompts_desde_segmento(segmento, labels, img_rgb, presence, modo_info):
    H, W = img_rgb.shape[:2]
    template = template_bbox.sort_values("id_real").reset_index(drop=True)
    prompts = {}
    filas = []

    for orden, (cand, label_idx) in enumerate(zip(segmento, labels)):
        vertebra = ID_TO_VERTEBRA[label_idx + 1]
        trow = template[template["id_real"] == label_idx + 1].iloc[0]
        cx = cand["x"] / IMG_SIZE * W
        cy = cand["y"] / IMG_SIZE * H
        pred_w = np.clip(cand["wh_rel"][0], 0.03, 0.28) * W
        pred_h = np.clip(cand["wh_rel"][1], 0.025, 0.18) * H
        tpl_w = float(trow["w_rel"] * W)
        tpl_h = float(trow["h_rel"] * H)
        bw = (0.65 * pred_w + 0.35 * tpl_w) * BOX_EXPAND_W
        bh = (0.65 * pred_h + 0.35 * tpl_h) * BOX_EXPAND_H
        bbox = bbox_clip([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], H, W)

        prompts[vertebra] = {
            "vertebra": vertebra,
            "id_real": label_idx + 1,
            "bbox_xyxy": bbox,
            "confianza_nn": float(cand["score"]),
            "confianza_original": float(cand.get("score_original", cand["score"])),
            "presencia_nn": float(presence[label_idx]) if label_idx < len(presence) else np.nan,
            "estrategia_etiquetado": modo_info["modo_solicitado"],
            "estrategia_resuelta": modo_info["modo_resuelto"],
            "orden_caja": int(orden),
            "n_camino": int(modo_info["n_camino"]),
            "y_min_anatomico": float(cand.get("y_min_anatomico", 0.0)),
            "prompt_origen": "cajas_nn_centernet_lite",
        }
        filas.append({
            "vertebra": vertebra,
            "id_real": label_idx + 1,
            "cx": cx,
            "cy": cy,
            "confianza": float(cand["score"]),
            "confianza_original": float(cand.get("score_original", cand["score"])),
            "presencia": float(presence[label_idx]) if label_idx < len(presence) else np.nan,
            "penalizado_craneo": bool(cand.get("penalizado_craneo", False)),
            "y_min_anatomico": float(cand.get("y_min_anatomico", 0.0)),
            "bbox_xyxy": bbox,
            "estrategia_etiquetado": modo_info["modo_solicitado"],
            "estrategia_resuelta": modo_info["modo_resuelto"],
            "orden_caja": int(orden),
            "n_camino": int(modo_info["n_camino"]),
        })

    return prompts, pd.DataFrame(filas).sort_values("id_real").reset_index(drop=True)


def prompts_desde_outputs(img_rgb, out, modo_etiquetado=None):
    heat, wh_map, off_map, presence = out["heat"], out["wh"], out["off"], out["presence"]
    picos = extraer_picos(heat)
    y_min_hm, info_anatomico = estimar_y_min_anatomico(img_rgb)

    candidatos = []
    for p in picos:
        x, y = p["x"], p["y"]
        cx_hm = x + float(off_map[0, y, x])
        cy_hm = y + float(off_map[1, y, x])
        score = float(p["score"])

        if cy_hm < y_min_hm:
            if cy_hm < y_min_hm - IMG_SIZE * 0.06:
                continue
            score *= CRANEO_SCORE_FACTOR

        candidatos.append({
            "x": cx_hm,
            "y": cy_hm,
            "score": score,
            "score_original": float(p["score"]),
            "penalizado_craneo": bool(cy_hm < y_min_hm),
            "y_min_anatomico": float(y_min_hm),
            "wh_rel": [float(wh_map[0, y, x]), float(wh_map[1, y, x])],
        })

    camino = seleccionar_camino_dp(candidatos, n_pasos=N_CAJAS_CAMINO)
    segmento, labels, modo_info = elegir_segmento_y_labels(camino, modo_etiquetado=modo_etiquetado)
    prompts, df_centros = construir_prompts_desde_segmento(segmento, labels, img_rgb, presence, modo_info)

    df_candidatos = pd.DataFrame(candidatos)
    df_candidatos.attrs["info_anatomico"] = info_anatomico
    df_candidatos.attrs["modo_info"] = modo_info
    return prompts, df_centros, df_candidatos


def generar_prompts_nn(split, patient_id, modo_etiquetado=None):
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    img = cargar_imagen(path_img)
    mask = cargar_mascara(path_mask)
    out = predecir_outputs(img)
    prompts, df_centros, df_candidatos = prompts_desde_outputs(img, out, modo_etiquetado=modo_etiquetado)
    return img, mask, prompts, df_centros, df_candidatos, out


## 8. Metricas de cajas

Se conservan las metricas estricta, flexible y shift porque ayudan a separar dos problemas:

- si la vertebra fue encontrada espacialmente;
- si fue nombrada con la etiqueta anatomica correcta.

Esta distincion sigue siendo importante para optimizar la version final.


In [ ]:
# Metricas geometricas de caja contra mascara vertebral disponible.
def mask_binaria_vertebra(mask, vertebra):
    return (mask == VERTEBRA_TO_ID[vertebra]).astype(np.uint8)


def vertebras_gt_eval(mascara_gt):
    return vertebras_presentes_en_mascara(mascara_gt)


def metricas_bbox_contra_gt(gt, bbox):
    x0, y0, x1, y1 = [int(v) for v in bbox]
    cover = np.zeros_like(gt, dtype=bool)
    cover[y0:y1 + 1, x0:x1 + 1] = True
    inter = int(np.logical_and(gt, cover).sum())
    union = int(np.logical_or(gt, cover).sum())
    bbox_area = int(cover.sum())
    total_gt = int(gt.sum())

    ys, xs = np.where(gt)
    if len(xs) == 0:
        center_error = np.nan
    else:
        cx_gt, cy_gt = xs.mean(), ys.mean()
        cx_box, cy_box = (x0 + x1) / 2, (y0 + y1) / 2
        center_error = math.sqrt((cx_box - cx_gt) ** 2 + (cy_box - cy_gt) ** 2)

    return {
        "bbox_iou": inter / union if union else 0.0,
        "bbox_recall": inter / total_gt if total_gt else 0.0,
        "bbox_precision": inter / bbox_area if bbox_area else 0.0,
        "center_error_px": center_error,
    }


def mejor_prompt_para_gt(prompts_auto, gt):
    mejor = None
    mejor_key = None
    for vertebra_prompt, info in prompts_auto.items():
        if "bbox_xyxy" not in info:
            continue
        met = metricas_bbox_contra_gt(gt, info["bbox_xyxy"])
        center = met["center_error_px"]
        center_key = -center if np.isfinite(center) else -1e9
        key = (met["bbox_iou"], met["bbox_recall"], center_key)
        if mejor is None or key > mejor_key:
            mejor = {
                "prompt_flexible_vertebra": vertebra_prompt,
                "prompt_flexible_id": VERTEBRA_TO_ID.get(vertebra_prompt, np.nan),
                "bbox_iou_flexible": met["bbox_iou"],
                "bbox_recall_flexible": met["bbox_recall"],
                "bbox_precision_flexible": met["bbox_precision"],
                "center_error_flexible_px": met["center_error_px"],
                "confianza_flexible": info.get("confianza_nn", np.nan),
            }
            mejor_key = key
    return mejor


# Evalua cajas estrictas y tambien mejor correspondencia flexible por vertebra GT.
def evaluar_cajas(prompts_auto, mascara_gt, vertebras_eval=None):
    if vertebras_eval is None:
        vertebras_eval = vertebras_gt_eval(mascara_gt)

    filas = []
    for vertebra in vertebras_eval:
        gt = mask_binaria_vertebra(mascara_gt, vertebra).astype(bool)
        total_gt = int(gt.sum())
        if total_gt == 0:
            continue

        mejor = mejor_prompt_para_gt(prompts_auto, gt)
        if vertebra in prompts_auto:
            met_exacta = metricas_bbox_contra_gt(gt, prompts_auto[vertebra]["bbox_xyxy"])
            estado_prompt = "evaluado_gt"
            confianza_nn = prompts_auto[vertebra].get("confianza_nn", np.nan)
            presencia_nn = prompts_auto[vertebra].get("presencia_nn", np.nan)
        else:
            met_exacta = {
                "bbox_iou": 0.0,
                "bbox_recall": 0.0,
                "bbox_precision": 0.0,
                "center_error_px": np.nan,
            }
            estado_prompt = "gt_sin_prompt"
            confianza_nn = np.nan
            presencia_nn = np.nan

        fila = {
            "vertebra": vertebra,
            "id_real": VERTEBRA_TO_ID[vertebra],
            "bbox_iou": met_exacta["bbox_iou"],
            "bbox_recall": met_exacta["bbox_recall"],
            "bbox_precision": met_exacta["bbox_precision"],
            "center_error_px": met_exacta["center_error_px"],
            "confianza_nn": confianza_nn,
            "presencia_nn": presencia_nn,
            "estado_prompt": estado_prompt,
        }

        if mejor is not None:
            fila.update(mejor)
            fila["flexible_misma_etiqueta"] = bool(mejor["prompt_flexible_vertebra"] == vertebra)
            fila["desfase_id_flexible"] = int(mejor["prompt_flexible_id"] - VERTEBRA_TO_ID[vertebra])
            fila["mejora_iou_flexible"] = float(mejor["bbox_iou_flexible"] - met_exacta["bbox_iou"])
        else:
            fila.update({
                "prompt_flexible_vertebra": "",
                "prompt_flexible_id": np.nan,
                "bbox_iou_flexible": np.nan,
                "bbox_recall_flexible": np.nan,
                "bbox_precision_flexible": np.nan,
                "center_error_flexible_px": np.nan,
                "confianza_flexible": np.nan,
                "flexible_misma_etiqueta": False,
                "desfase_id_flexible": np.nan,
                "mejora_iou_flexible": np.nan,
            })

        filas.append(fila)

    if not filas:
        return pd.DataFrame(columns=[
            "vertebra", "id_real", "bbox_iou", "bbox_recall", "bbox_precision",
            "center_error_px", "confianza_nn", "presencia_nn", "estado_prompt",
            "prompt_flexible_vertebra", "prompt_flexible_id", "bbox_iou_flexible",
            "bbox_recall_flexible", "bbox_precision_flexible", "center_error_flexible_px",
            "flexible_misma_etiqueta", "desfase_id_flexible", "mejora_iou_flexible",
        ])

    return pd.DataFrame(filas).sort_values("id_real").reset_index(drop=True)


def resumen_etiquetas_prompts(prompts_auto, vertebras_gt):
    labels_prompts = sorted([v for v in prompts_auto.keys() if v in VERTEBRA_TO_ID], key=lambda v: VERTEBRA_TO_ID[v])
    labels_gt = list(vertebras_gt)
    labels_eval = [v for v in labels_prompts if v in labels_gt]
    labels_extra = [v for v in labels_prompts if v not in labels_gt]
    labels_faltantes = [v for v in labels_gt if v not in labels_prompts]
    return labels_prompts, labels_eval, labels_extra, labels_faltantes


def describir_desfases_flexibles(df, min_mejora=0.05):
    if df.empty or "prompt_flexible_vertebra" not in df:
        return ""
    filas = []
    for _, row in df.iterrows():
        prompt = row.get("prompt_flexible_vertebra", "")
        if not prompt or prompt == row["vertebra"]:
            continue
        mejora = float(row.get("mejora_iou_flexible", 0.0))
        iou_flex = float(row.get("bbox_iou_flexible", 0.0))
        if mejora >= min_mejora:
            filas.append(f"{row['vertebra']}<-{prompt}({iou_flex:.2f})")
    return ";".join(filas)


def evaluar_muestra_nn(split, patient_id, modo_etiquetado=None):
    t0 = time.perf_counter()
    img, mask, prompts, df_centros, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=modo_etiquetado)
    vertebras_gt = vertebras_gt_eval(mask)
    df = evaluar_cajas(prompts, mask, vertebras_eval=vertebras_gt)
    labels_prompts, labels_eval, labels_extra, labels_faltantes = resumen_etiquetas_prompts(prompts, vertebras_gt)
    tipo_real = "escoliosis" if str(patient_id).startswith("S_") else "normal"

    return {
        "split": split,
        "patient_id": patient_id,
        "tipo_real": tipo_real,
        "modo_evaluacion": "estricta_y_flexible_solo_etiquetas_gt",
        "estrategia_etiquetado": df_centros["estrategia_etiquetado"].iloc[0] if not df_centros.empty else (modo_etiquetado or ETIQUETADO_MODO),
        "estrategia_resuelta": df_centros["estrategia_resuelta"].iloc[0] if not df_centros.empty else "",
        "n_camino": int(df_centros["n_camino"].iloc[0]) if not df_centros.empty and "n_camino" in df_centros else 0,
        "n_gt": int(len(vertebras_gt)),
        "n_prompts": int(len(prompts)),
        "n_prompts_evaluados": int(len(labels_eval)),
        "n_prompts_extra_no_evaluados": int(len(labels_extra)),
        "n_gt_sin_prompt": int(len(labels_faltantes)),
        "labels_gt": ",".join(vertebras_gt),
        "labels_prompts_evaluados": ",".join(labels_eval),
        "labels_prompts_extra_no_evaluados": ",".join(labels_extra),
        "labels_gt_sin_prompt": ",".join(labels_faltantes),
        "tiempo_s": time.perf_counter() - t0,
        "bbox_iou_promedio": float(df["bbox_iou"].mean()) if not df.empty else np.nan,
        "bbox_recall_promedio": float(df["bbox_recall"].mean()) if not df.empty else np.nan,
        "bbox_precision_promedio": float(df["bbox_precision"].mean()) if not df.empty else np.nan,
        "n_vertebras_iou_mayor_02": int((df["bbox_iou"] > 0.20).sum()) if not df.empty else 0,
        "center_error_px": float(df["center_error_px"].mean()) if not df.empty else np.nan,
        "bbox_iou_flexible_promedio": float(df["bbox_iou_flexible"].mean()) if not df.empty else np.nan,
        "bbox_recall_flexible_promedio": float(df["bbox_recall_flexible"].mean()) if not df.empty else np.nan,
        "bbox_precision_flexible_promedio": float(df["bbox_precision_flexible"].mean()) if not df.empty else np.nan,
        "n_vertebras_flexible_iou_mayor_02": int((df["bbox_iou_flexible"] > 0.20).sum()) if not df.empty else 0,
        "center_error_flexible_px": float(df["center_error_flexible_px"].mean()) if not df.empty else np.nan,
        "n_flexible_misma_etiqueta": int(df["flexible_misma_etiqueta"].sum()) if not df.empty else 0,
        "mejora_iou_flexible_promedio": float(df["mejora_iou_flexible"].mean()) if not df.empty else np.nan,
        "desfase_id_flexible_promedio": float(df["desfase_id_flexible"].mean()) if not df.empty else np.nan,
        "desfases_flexibles": describir_desfases_flexibles(df),
        "confianza_media": float(df["confianza_nn"].mean()) if "confianza_nn" in df else np.nan,
    }, df


def evaluar_split_nn(split="val", patient_ids=None, modo_etiquetado=None):
    if patient_ids is None:
        patient_ids = sorted(PROMPTS_DICC[split].keys())

    resumen, detalles, errores = [], [], []
    for pid in tqdm(patient_ids, desc=f"Evaluando {split}"):
        try:
            row, df = evaluar_muestra_nn(split, pid, modo_etiquetado=modo_etiquetado)
            resumen.append(row)
            detalles.append(df.assign(split=split, patient_id=pid))
        except Exception as exc:
            errores.append({"split": split, "patient_id": pid, "error": repr(exc)})

    df_resumen = pd.DataFrame(resumen)
    df_detalle = pd.concat(detalles, ignore_index=True) if detalles else pd.DataFrame()
    df_errores = pd.DataFrame(errores)

    df_resumen.to_csv(RESULTADOS_DIR / f"cajas_nn_centernet_val_resumen.csv", index=False)
    df_detalle.to_csv(RESULTADOS_DIR / f"cajas_nn_centernet_val_detalle.csv", index=False)
    df_errores.to_csv(RESULTADOS_DIR / f"cajas_nn_centernet_val_errores.csv", index=False)
    return df_resumen, df_detalle, df_errores



def comparar_estrategias_etiquetado(split="val", patient_ids=None, estrategias=None):
    """Diagnostico opcional. La ruta principal usa solo top_anchor."""
    estrategias = estrategias or ESTRATEGIAS_ETIQUETADO
    frames = []
    for estrategia in estrategias:
        df_res, _, _ = evaluar_split_nn(split, patient_ids=patient_ids, modo_etiquetado=estrategia)
        if not df_res.empty:
            frames.append(df_res)

    df_cmp = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    out_path = RESULTADOS_DIR / f"cajas_nn_centernet_{split}_comparativo_estrategias.csv"
    df_cmp.to_csv(out_path, index=False)
    print("Comparativo guardado:", out_path)
    return df_cmp


## 9. Evaluacion de cajas con shift

Evalua la red de cajas y estima corrimientos anatomicos. No es una estrategia alternativa de segmentacion; es diagnostico necesario para entender errores en escoliosis o GT parcial.


In [ ]:
# Casos trazadores usados para revisar visualmente normales y escoliosis.
PACIENTES_PRUEBA = ["N_12", "N_32", "S_187", "S_130", "S_80", "S_190"]
SHIFTS_ANATOMICOS = list(range(-6, 7))
IOU_PROMPT_UTIL = 0.20
RECALL_PROMPT_UTIL = 0.70


# Corrige un desfase anatomico uniforme sin mover las cajas.
def relabel_prompts_por_shift(prompts_auto, shift):
    """Desplaza nombres: si shift=+1, una caja T2 pasa a evaluarse como T1."""
    corregidos = {}
    asignaciones = []
    for old_label, info in prompts_auto.items():
        old_id = VERTEBRA_TO_ID.get(old_label)
        if old_id is None:
            continue
        new_id = old_id - int(shift)
        if new_id < 1 or new_id > N_CLASES:
            continue
        new_label = ID_TO_VERTEBRA[new_id]
        nuevo = dict(info)
        nuevo["vertebra_original"] = old_label
        nuevo["vertebra"] = new_label
        nuevo["id_real_original"] = old_id
        nuevo["id_real"] = new_id
        nuevo["shift_anatomico"] = int(shift)
        nuevo["prompt_origen"] = "cajas_nn_centernet_lite_shift"
        if new_label not in corregidos or nuevo.get("confianza_nn", 0) > corregidos[new_label].get("confianza_nn", 0):
            corregidos[new_label] = nuevo
        asignaciones.append({"vertebra_original": old_label, "vertebra_corregida": new_label, "shift": int(shift)})
    return corregidos, pd.DataFrame(asignaciones)


# Prueba varios shifts y escoge el que mejora recall/IoU contra GT disponible.
def evaluar_shifts_anatomicos(prompts_auto, mascara_gt, shifts=SHIFTS_ANATOMICOS):
    filas = []
    vertebras_gt = vertebras_gt_eval(mascara_gt)
    for shift in shifts:
        prompts_shift, _ = relabel_prompts_por_shift(prompts_auto, shift)
        df_shift = evaluar_cajas(prompts_shift, mascara_gt, vertebras_eval=vertebras_gt)
        filas.append({
            "shift": int(shift),
            "n_prompts_shift": int(len(prompts_shift)),
            "bbox_iou_shift": float(df_shift["bbox_iou"].mean()) if not df_shift.empty else np.nan,
            "bbox_recall_shift": float(df_shift["bbox_recall"].mean()) if not df_shift.empty else np.nan,
            "center_error_shift_px": float(df_shift["center_error_px"].mean()) if not df_shift.empty else np.nan,
            "n_vertebras_iou_mayor_02_shift": int((df_shift["bbox_iou"] > IOU_PROMPT_UTIL).sum()) if not df_shift.empty else 0,
            "n_vertebras_recall_mayor_07_shift": int((df_shift["bbox_recall"] > RECALL_PROMPT_UTIL).sum()) if not df_shift.empty else 0,
        })

    df_shifts = pd.DataFrame(filas)
    if df_shifts.empty:
        return 0, df_shifts, pd.DataFrame()

    orden = df_shifts.sort_values(
        ["bbox_recall_shift", "bbox_iou_shift", "n_vertebras_iou_mayor_02_shift", "shift"],
        ascending=[False, False, False, True],
    )
    best_shift = int(orden.iloc[0]["shift"])
    prompts_best, df_asignaciones = relabel_prompts_por_shift(prompts_auto, best_shift)
    df_best = evaluar_cajas(prompts_best, mascara_gt, vertebras_eval=vertebras_gt)
    return best_shift, df_shifts, df_best


def evaluar_muestra_nn_sam(split, patient_id, modo_etiquetado=None):
    t0 = time.perf_counter()
    img, mask, prompts, df_centros, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=modo_etiquetado)
    vertebras_gt = vertebras_gt_eval(mask)
    df_base = evaluar_cajas(prompts, mask, vertebras_eval=vertebras_gt)
    best_shift, df_shifts, df_shift = evaluar_shifts_anatomicos(prompts, mask)
    labels_prompts, labels_eval, labels_extra, labels_faltantes = resumen_etiquetas_prompts(prompts, vertebras_gt)
    tipo_real = "escoliosis" if str(patient_id).startswith("S_") else "normal"

    mejor_shift_row = df_shifts[df_shifts["shift"] == best_shift].iloc[0].to_dict() if not df_shifts.empty else {}
    return {
        "split": split,
        "patient_id": patient_id,
        "tipo_real": tipo_real,
        "estrategia_etiquetado": df_centros["estrategia_etiquetado"].iloc[0] if not df_centros.empty else (modo_etiquetado or ETIQUETADO_MODO),
        "estrategia_resuelta": df_centros["estrategia_resuelta"].iloc[0] if not df_centros.empty else "",
        "n_gt": int(len(vertebras_gt)),
        "n_prompts": int(len(prompts)),
        "n_prompts_extra_no_evaluados": int(len(labels_extra)),
        "labels_gt": ",".join(vertebras_gt),
        "labels_prompts_extra_no_evaluados": ",".join(labels_extra),
        "tiempo_cajas_s": time.perf_counter() - t0,
        "bbox_iou_estricto": float(df_base["bbox_iou"].mean()) if not df_base.empty else np.nan,
        "bbox_recall_estricto": float(df_base["bbox_recall"].mean()) if not df_base.empty else np.nan,
        "center_error_estricto_px": float(df_base["center_error_px"].mean()) if not df_base.empty else np.nan,
        "bbox_iou_flexible": float(df_base["bbox_iou_flexible"].mean()) if not df_base.empty else np.nan,
        "bbox_recall_flexible": float(df_base["bbox_recall_flexible"].mean()) if not df_base.empty else np.nan,
        "center_error_flexible_px": float(df_base["center_error_flexible_px"].mean()) if not df_base.empty else np.nan,
        "n_prompt_util_iou_flexible": int((df_base["bbox_iou_flexible"] > IOU_PROMPT_UTIL).sum()) if not df_base.empty else 0,
        "n_prompt_util_recall_flexible": int((df_base["bbox_recall_flexible"] > RECALL_PROMPT_UTIL).sum()) if not df_base.empty else 0,
        "best_shift": best_shift,
        "bbox_iou_shift": float(mejor_shift_row.get("bbox_iou_shift", np.nan)),
        "bbox_recall_shift": float(mejor_shift_row.get("bbox_recall_shift", np.nan)),
        "center_error_shift_px": float(mejor_shift_row.get("center_error_shift_px", np.nan)),
        "n_vertebras_iou_mayor_02_shift": int(mejor_shift_row.get("n_vertebras_iou_mayor_02_shift", 0)),
        "n_vertebras_recall_mayor_07_shift": int(mejor_shift_row.get("n_vertebras_recall_mayor_07_shift", 0)),
        "desfases_flexibles": describir_desfases_flexibles(df_base),
    }, df_base, df_shifts, df_shift


def evaluar_split_nn_sam(split="val", patient_ids=None):
    if patient_ids is None:
        patient_ids = sorted(PROMPTS_DICC[split].keys())

    resumen, detalles_base, detalles_shift, curvas_shift, errores = [], [], [], [], []
    for pid in tqdm(patient_ids, desc=f"NN-SAM {split}"):
        try:
            row, df_base, df_shifts, df_shift = evaluar_muestra_nn_sam(split, pid, modo_etiquetado=ETIQUETADO_MODO)
            resumen.append(row)
            detalles_base.append(df_base.assign(split=split, patient_id=pid))
            detalles_shift.append(df_shift.assign(split=split, patient_id=pid, best_shift=row["best_shift"]))
            curvas_shift.append(df_shifts.assign(split=split, patient_id=pid))
        except Exception as exc:
            errores.append({"split": split, "patient_id": pid, "error": repr(exc)})

    df_resumen = pd.DataFrame(resumen)
    df_base_all = pd.concat(detalles_base, ignore_index=True) if detalles_base else pd.DataFrame()
    df_shift_all = pd.concat(detalles_shift, ignore_index=True) if detalles_shift else pd.DataFrame()
    df_shifts_all = pd.concat(curvas_shift, ignore_index=True) if curvas_shift else pd.DataFrame()
    df_errores = pd.DataFrame(errores)

    df_resumen.to_csv(NN_SAM_DIR / f"nn_sam_{split}_resumen.csv", index=False)
    df_base_all.to_csv(NN_SAM_DIR / f"nn_sam_{split}_detalle_base.csv", index=False)
    df_shift_all.to_csv(NN_SAM_DIR / f"nn_sam_{split}_detalle_shift.csv", index=False)
    df_shifts_all.to_csv(NN_SAM_DIR / f"nn_sam_{split}_curvas_shift.csv", index=False)
    df_errores.to_csv(NN_SAM_DIR / f"nn_sam_{split}_errores.csv", index=False)
    return df_resumen, df_base_all, df_shift_all, df_shifts_all, df_errores


## 10. Evaluacion de cajas en validacion y test

Mide si NN-SAM genera prompts suficientemente buenos antes de pasar por MedSAM. Esta etapa se conserva porque permite saber si un fallo posterior viene de las cajas o del segmentador.


In [ ]:
patient_ids_eval = sorted(PROMPTS_DICC["val"].keys()) if EVALUAR_TODO_VAL else [p for p in PACIENTES_PRUEBA if p in PROMPTS_DICC["val"]]
df_nn_sam_resumen, df_nn_sam_detalle_base, df_nn_sam_detalle_shift, df_nn_sam_curvas_shift, df_nn_sam_errores = evaluar_split_nn_sam("val", patient_ids_eval)

display(df_nn_sam_resumen)

if not df_nn_sam_resumen.empty:
    display(
        df_nn_sam_resumen.groupby("tipo_real", as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            tiempo_cajas_s=("tiempo_cajas_s", "mean"),
            iou_estricto=("bbox_iou_estricto", "mean"),
            recall_estricto=("bbox_recall_estricto", "mean"),
            iou_flexible=("bbox_iou_flexible", "mean"),
            recall_flexible=("bbox_recall_flexible", "mean"),
            iou_shift=("bbox_iou_shift", "mean"),
            recall_shift=("bbox_recall_shift", "mean"),
            prompt_util_iou=("n_prompt_util_iou_flexible", "mean"),
            prompt_util_recall=("n_prompt_util_recall_flexible", "mean"),
        )
    )

display(df_nn_sam_errores)

if EVALUAR_TEST_FINAL and "test" in PROMPTS_DICC and len(PROMPTS_DICC["test"]) > 0:
    patient_ids_test = sorted(PROMPTS_DICC["test"].keys())
    df_nn_sam_test_resumen, df_nn_sam_test_detalle_base, df_nn_sam_test_detalle_shift, df_nn_sam_test_curvas_shift, df_nn_sam_test_errores = evaluar_split_nn_sam("test", patient_ids_test)
    display(df_nn_sam_test_resumen)
    display(df_nn_sam_test_errores)


## 11. Visualizacion de cajas

Visualizacion diagnostica de cajas estrictas, flexibles y con shift. Se conserva para revisar cualitativamente los casos dificiles, pero no introduce una estrategia alternativa.


In [ ]:
def dibujar_bbox(ax, bbox, label, color, linewidth=1.2):
    x0, y0, x1, y1 = bbox
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=linewidth))
    ax.text(x0, max(0, y0 - 3), label, fontsize=7, color="white", bbox=dict(facecolor="black", alpha=0.45, pad=1))


def visualizar_nn_sam(split, patient_id, guardar=True):
    img, mask, prompts, df_centros, df_candidatos, out = generar_prompts_nn(split, patient_id, modo_etiquetado=ETIQUETADO_MODO)
    vertebras_gt = vertebras_gt_eval(mask)
    df_base = evaluar_cajas(prompts, mask, vertebras_eval=vertebras_gt)
    best_shift, df_shifts, df_shift = evaluar_shifts_anatomicos(prompts, mask)
    prompts_shift, df_asig = relabel_prompts_por_shift(prompts, best_shift)

    tab_base = df_base.set_index("vertebra") if not df_base.empty else pd.DataFrame()
    tab_shift = df_shift.set_index("vertebra") if not df_shift.empty else pd.DataFrame()
    overlay = np.zeros_like(img)
    overlay[..., 0] = (mask > 0).astype(np.uint8) * 255

    fig, ax = plt.subplots(1, 3, figsize=(22, 9))
    titulos = [
        "Estricta: misma etiqueta",
        "Flexible: mejor caja para cada GT",
        f"Shift anatomico: {best_shift:+d}",
    ]
    for a, titulo in zip(ax, titulos):
        a.imshow(img)
        a.imshow(overlay, alpha=0.16)
        a.set_title(titulo)
        a.axis("off")

    for vertebra, info in prompts.items():
        if vertebra not in vertebras_gt:
            continue
        iou = float(tab_base.loc[vertebra, "bbox_iou"]) if vertebra in tab_base.index else 0.0
        dibujar_bbox(ax[0], info["bbox_xyxy"], f"{vertebra} {iou:.2f}", "lime")

    for _, row in df_base.iterrows():
        gt_v = row["vertebra"]
        best_v = row.get("prompt_flexible_vertebra", "")
        if not best_v or best_v not in prompts:
            continue
        color = "lime" if best_v == gt_v else "gold"
        dibujar_bbox(ax[1], prompts[best_v]["bbox_xyxy"], f"GT {gt_v}<-{best_v} {float(row['bbox_iou_flexible']):.2f}", color)

    for vertebra, info in prompts_shift.items():
        if vertebra not in vertebras_gt:
            continue
        iou = float(tab_shift.loc[vertebra, "bbox_iou"]) if vertebra in tab_shift.index else 0.0
        old = info.get("vertebra_original", vertebra)
        dibujar_bbox(ax[2], info["bbox_xyxy"], f"{vertebra}<-{old} {iou:.2f}", "cyan")

    iou_gt = float(df_base["bbox_iou"].mean()) if not df_base.empty else np.nan
    iou_fx = float(df_base["bbox_iou_flexible"].mean()) if not df_base.empty else np.nan
    iou_sh = float(df_shift["bbox_iou"].mean()) if not df_shift.empty else np.nan
    recall_fx = float(df_base["bbox_recall_flexible"].mean()) if not df_base.empty else np.nan
    recall_sh = float(df_shift["bbox_recall"].mean()) if not df_shift.empty else np.nan
    fig.suptitle(
        f"{patient_id} | IoU estricto={iou_gt:.3f} | IoU flex={iou_fx:.3f} | IoU shift={iou_sh:.3f} "
        f"| recall flex={recall_fx:.3f} | recall shift={recall_sh:.3f} | GT={len(vertebras_gt)}",
        fontsize=13,
    )
    plt.tight_layout()

    if guardar:
        out_path = NN_SAM_DIR / f"visual_nn_sam_{split}_{patient_id}_shift_{best_shift:+d}.png"
        plt.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    display(df_centros)
    display(df_shifts.sort_values(["bbox_recall_shift", "bbox_iou_shift"], ascending=False).head(8))
    display(pd.DataFrame([{
        "patient_id": patient_id,
        "best_shift": best_shift,
        "labels_gt": ",".join(vertebras_gt),
        "desfases_flexibles": describir_desfases_flexibles(df_base),
    }]))
    display(df_base)
    display(df_shift)
    return prompts, prompts_shift, df_base, df_shift, df_shifts


for pid in [p for p in PACIENTES_VISUALIZACION if p in PROMPTS_DICC["val"]]:
    visualizar_nn_sam("val", pid)


## 12. Red neuronal de nombres anatomicos

Las cajas automaticas ya estaban encontrando la region vertebral con buen rendimiento flexible, pero parte del error estricto venia de asignar mal el nombre anatomico. Esta seccion entrena una red ligera sobre la secuencia de cajas detectadas: no segmenta de nuevo, solo aprende si una caja corresponde a T1-L5 o si debe tratarse como una caja extra.

La idea es aprovechar lo que funciono: NN-SAM propone la ruta de la columna y MedSAM segmenta. Esta red intermedia intenta corregir el desfase anatomico antes de llamar a MedSAM.


In [ ]:
FEATURES_NOMBRES = [
    "rank_rel",
    "n_prompts_rel",
    "cx_rel",
    "cy_rel",
    "w_rel",
    "h_rel",
    "area_rel",
    "aspect",
    "dy_prev_rel",
    "dy_next_rel",
    "dx_prev_rel",
    "dx_next_rel",
    "confianza_nn",
    "confianza_original",
    "presencia_nn",
]
NOMBRES_PAD_ID = -100
NOMBRES_N_CLASSES = N_CLASES + 1  # 17 vertebras + clase extra/sin etiqueta.


def ordenar_prompts_vertical(prompts_auto):
    """Ordena cajas de superior a inferior sin depender del nombre previo asignado."""
    return sorted(
        prompts_auto.items(),
        key=lambda kv: ((float(kv[1]["bbox_xyxy"][1]) + float(kv[1]["bbox_xyxy"][3])) / 2.0,
                        (float(kv[1]["bbox_xyxy"][0]) + float(kv[1]["bbox_xyxy"][2])) / 2.0),
    )


def extraer_features_secuencia_cajas(prompts_auto, img_rgb):
    """Convierte la secuencia de cajas NN-SAM en features geometricos y de confianza."""
    H, W = img_rgb.shape[:2]
    items = ordenar_prompts_vertical(prompts_auto)
    n = len(items)
    centros = []
    for _, info in items:
        x0, y0, x1, y1 = [float(v) for v in info["bbox_xyxy"]]
        centros.append(((x0 + x1) / 2.0, (y0 + y1) / 2.0))

    rows = []
    for i, (raw_label, info) in enumerate(items):
        x0, y0, x1, y1 = [float(v) for v in info["bbox_xyxy"]]
        cx, cy = centros[i]
        bw = max(x1 - x0, 1.0)
        bh = max(y1 - y0, 1.0)
        if i == 0:
            dx_prev = 0.0
            dy_prev = 0.0
        else:
            dx_prev = cx - centros[i - 1][0]
            dy_prev = cy - centros[i - 1][1]
        if i == n - 1:
            dx_next = 0.0
            dy_next = 0.0
        else:
            dx_next = centros[i + 1][0] - cx
            dy_next = centros[i + 1][1] - cy

        row = {
            "raw_label": raw_label,
            "rank": i,
            "rank_rel": i / max(n - 1, 1),
            "n_prompts_rel": n / max(N_CAJAS_CAMINO, 1),
            "cx_rel": cx / max(W, 1),
            "cy_rel": cy / max(H, 1),
            "w_rel": bw / max(W, 1),
            "h_rel": bh / max(H, 1),
            "area_rel": (bw * bh) / max(W * H, 1),
            "aspect": bw / max(bh, 1.0),
            "dy_prev_rel": dy_prev / max(H, 1),
            "dy_next_rel": dy_next / max(H, 1),
            "dx_prev_rel": dx_prev / max(W, 1),
            "dx_next_rel": dx_next / max(W, 1),
            "confianza_nn": float(info.get("confianza_nn", info.get("confianza", 0.0))),
            "confianza_original": float(info.get("confianza_original", info.get("confianza_nn", 0.0))),
            "presencia_nn": float(info.get("presencia_nn", 0.0)) if np.isfinite(float(info.get("presencia_nn", 0.0))) else 0.0,
            "bbox_xyxy": [int(round(v)) for v in [x0, y0, x1, y1]],
        }
        rows.append(row)
    return rows


def target_nombre_para_bbox(mask_gt, bbox):
    """Asigna etiqueta docente a una caja usando solo las vertebras realmente anotadas en la mascara."""
    mejor = {"target_id": NOMBRES_EXTRA_ID, "target_label": NOMBRES_EXTRA_LABEL, "iou": 0.0, "recall": 0.0}
    for vertebra in vertebras_gt_eval(mask_gt):
        gt = mask_binaria_vertebra(mask_gt, vertebra).astype(bool)
        met = metricas_bbox_contra_gt(gt, bbox)
        key = (met["bbox_iou"], met["bbox_recall"])
        if key > (mejor["iou"], mejor["recall"]):
            mejor = {
                "target_id": VERTEBRA_TO_ID[vertebra] - 1,
                "target_label": vertebra,
                "iou": float(met["bbox_iou"]),
                "recall": float(met["bbox_recall"]),
            }
    if mejor["iou"] >= NOMBRES_MIN_IOU_TARGET or mejor["recall"] >= NOMBRES_MIN_RECALL_TARGET:
        return mejor
    mejor["target_id"] = NOMBRES_EXTRA_ID
    mejor["target_label"] = NOMBRES_EXTRA_LABEL
    return mejor


def construir_dataset_nombres(split="train", patient_ids=None, regenerar=None):
    """Genera pares caja->nombre anatomico a partir de las cajas NN-SAM actuales."""
    regenerar = NOMBRES_REGENERAR_DATASET if regenerar is None else bool(regenerar)
    cache_path = NN_SAM_DIR / f"dataset_red_nombres_{split}.csv"
    if cache_path.exists() and not regenerar:
        return pd.read_csv(cache_path)

    patient_ids = patient_ids or sorted(PROMPTS_DICC[split].keys())
    filas, errores = [], []
    for patient_id in tqdm(patient_ids, desc=f"dataset nombres {split}"):
        try:
            img, mask_gt, prompts_auto, _, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=ETIQUETADO_MODO)
            feat_rows = extraer_features_secuencia_cajas(prompts_auto, img)
            for row in feat_rows:
                target = target_nombre_para_bbox(mask_gt, row["bbox_xyxy"])
                filas.append({
                    "split": split,
                    "patient_id": patient_id,
                    "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
                    **{k: row[k] for k in FEATURES_NOMBRES},
                    "raw_label": row["raw_label"],
                    "rank": row["rank"],
                    "target_id": int(target["target_id"]),
                    "target_label": target["target_label"],
                    "target_iou": float(target["iou"]),
                    "target_recall": float(target["recall"]),
                    "bbox_xyxy": json.dumps(row["bbox_xyxy"]),
                })
        except Exception as exc:
            errores.append({"split": split, "patient_id": patient_id, "error": repr(exc)})

    df = pd.DataFrame(filas)
    df.to_csv(cache_path, index=False)
    if errores:
        pd.DataFrame(errores).to_csv(NN_SAM_DIR / f"dataset_red_nombres_{split}_errores.csv", index=False)
    print(f"Dataset nombres {split}: {len(df)} cajas | cache: {cache_path}")
    return df


class RedNombresAnatomicos(nn.Module):
    """Red secuencial ligera para etiquetar cajas vertebrales ordenadas de arriba hacia abajo."""
    def __init__(self, n_features, hidden=NOMBRES_HIDDEN, n_classes=NOMBRES_N_CLASSES):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.gru = nn.GRU(hidden, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x, lengths=None):
        z = self.encoder(x)
        if lengths is not None:
            lengths_cpu = lengths.detach().cpu().clamp(min=1)
            packed = nn.utils.rnn.pack_padded_sequence(z, lengths_cpu, batch_first=True, enforce_sorted=False)
            packed_out, _ = self.gru(packed)
            z, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=x.shape[1])
        else:
            z, _ = self.gru(z)
        return self.head(z)


def preparar_secuencias_nombres(df):
    seqs = []
    for patient_id, g in df.sort_values(["patient_id", "rank"]).groupby("patient_id"):
        x = g[FEATURES_NOMBRES].fillna(0.0).to_numpy(dtype=np.float32)
        y = g["target_id"].to_numpy(dtype=np.int64)
        seqs.append({"patient_id": patient_id, "x": x, "y": y})
    return seqs


def collate_nombres(batch):
    max_len = max(len(item["y"]) for item in batch)
    n_feat = batch[0]["x"].shape[1]
    x = torch.zeros(len(batch), max_len, n_feat, dtype=torch.float32)
    y = torch.full((len(batch), max_len), NOMBRES_PAD_ID, dtype=torch.long)
    lengths = torch.zeros(len(batch), dtype=torch.long)
    patient_ids = []
    for i, item in enumerate(batch):
        n = len(item["y"])
        x[i, :n] = torch.from_numpy(item["x"])
        y[i, :n] = torch.from_numpy(item["y"])
        lengths[i] = n
        patient_ids.append(item["patient_id"])
    return {"x": x, "y": y, "lengths": lengths, "patient_ids": patient_ids}


def normalizar_features_nombres(train_seqs, val_seqs):
    all_x = np.concatenate([s["x"] for s in train_seqs], axis=0)
    mean = all_x.mean(axis=0).astype(np.float32)
    std = (all_x.std(axis=0) + 1e-6).astype(np.float32)
    for seqs in [train_seqs, val_seqs]:
        for s in seqs:
            s["x"] = ((s["x"] - mean) / std).astype(np.float32)
    return mean, std


def pesos_clase_nombres(df_train):
    counts = df_train["target_id"].value_counts().reindex(range(NOMBRES_N_CLASSES), fill_value=1).to_numpy(dtype=np.float32)
    weights = counts.sum() / (counts * len(counts))
    weights = np.clip(weights, 0.35, 4.0)
    weights[NOMBRES_EXTRA_ID] = min(weights[NOMBRES_EXTRA_ID], 0.90)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


@torch.no_grad()
def evaluar_loader_nombres(model_nombres, loader, criterion):
    model_nombres.eval()
    total_loss, total_ok, total_n = 0.0, 0, 0
    for batch in loader:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        lengths = batch["lengths"].to(DEVICE)
        logits = model_nombres(x, lengths)
        loss = criterion(logits.reshape(-1, NOMBRES_N_CLASSES), y.reshape(-1))
        mask = y != NOMBRES_PAD_ID
        pred = logits.argmax(dim=-1)
        total_loss += float(loss.item()) * int(mask.sum().item())
        total_ok += int(((pred == y) & mask).sum().item())
        total_n += int(mask.sum().item())
    return {
        "loss": total_loss / max(total_n, 1),
        "acc": total_ok / max(total_n, 1),
        "n": total_n,
    }


def entrenar_red_nombres_anatomicos():
    df_train = construir_dataset_nombres("train", regenerar=NOMBRES_REGENERAR_DATASET)
    df_val = construir_dataset_nombres("val", regenerar=NOMBRES_REGENERAR_DATASET)

    train_seqs = preparar_secuencias_nombres(df_train)
    val_seqs = preparar_secuencias_nombres(df_val)
    mean, std = normalizar_features_nombres(train_seqs, val_seqs)

    train_loader = DataLoader(train_seqs, batch_size=NOMBRES_BATCH_SIZE, shuffle=True, collate_fn=collate_nombres)
    val_loader = DataLoader(val_seqs, batch_size=NOMBRES_BATCH_SIZE, shuffle=False, collate_fn=collate_nombres)

    model_nombres = RedNombresAnatomicos(len(FEATURES_NOMBRES)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=pesos_clase_nombres(df_train), ignore_index=NOMBRES_PAD_ID)
    optimizer = torch.optim.AdamW(model_nombres.parameters(), lr=NOMBRES_LR, weight_decay=NOMBRES_WEIGHT_DECAY)

    best_val = float("inf")
    best_epoch = -1
    patience = 0
    hist = []
    for epoch in range(1, NOMBRES_EPOCHS + 1):
        model_nombres.train()
        total_loss, total_n = 0.0, 0
        for batch in tqdm(train_loader, desc=f"Red nombres epoch {epoch}", leave=False):
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            lengths = batch["lengths"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model_nombres(x, lengths)
            loss = criterion(logits.reshape(-1, NOMBRES_N_CLASSES), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_nombres.parameters(), 1.0)
            optimizer.step()
            n_valid = int((y != NOMBRES_PAD_ID).sum().item())
            total_loss += float(loss.item()) * n_valid
            total_n += n_valid

        train_loss = total_loss / max(total_n, 1)
        val_metrics = evaluar_loader_nombres(model_nombres, val_loader, criterion)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc_caja": val_metrics["acc"],
            "val_n_cajas": val_metrics["n"],
        }
        hist.append(row)
        print(row)

        if val_metrics["loss"] < best_val - 1e-4:
            best_val = val_metrics["loss"]
            best_epoch = epoch
            patience = 0
            torch.save({
                "model_state_dict": model_nombres.state_dict(),
                "features": FEATURES_NOMBRES,
                "mean": mean,
                "std": std,
                "best_epoch": best_epoch,
                "best_val_loss": best_val,
            }, NOMBRES_CKPT_PATH)
        else:
            patience += 1
            if patience >= NOMBRES_PACIENCIA:
                print(f"Parada temprana red nombres en epoch {epoch}. Mejor epoch: {best_epoch}")
                break

    hist_df = pd.DataFrame(hist)
    hist_df.to_csv(NN_SAM_DIR / "hist_red_nombres_anatomicos.csv", index=False)
    return cargar_red_nombres_anatomicos(), hist_df


def cargar_red_nombres_anatomicos():
    if not NOMBRES_CKPT_PATH.exists():
        raise FileNotFoundError(f"No existe checkpoint red nombres: {NOMBRES_CKPT_PATH}")
    ckpt = torch.load(NOMBRES_CKPT_PATH, map_location=DEVICE)
    model_nombres = RedNombresAnatomicos(len(ckpt.get("features", FEATURES_NOMBRES))).to(DEVICE)
    model_nombres.load_state_dict(ckpt["model_state_dict"])
    model_nombres.eval()
    model_nombres.features = ckpt.get("features", FEATURES_NOMBRES)
    model_nombres.mean = np.asarray(ckpt.get("mean"), dtype=np.float32)
    model_nombres.std = np.asarray(ckpt.get("std"), dtype=np.float32)
    model_nombres.best_epoch = ckpt.get("best_epoch", None)
    model_nombres.best_val_loss = ckpt.get("best_val_loss", None)
    print("Red nombres cargada:", NOMBRES_CKPT_PATH, "best_epoch:", model_nombres.best_epoch)
    return model_nombres


if USAR_RED_NOMBRES_ANATOMICOS:
    if EJECUTAR_ENTRENAMIENTO_NOMBRES or not NOMBRES_CKPT_PATH.exists():
        naming_model_final, hist_red_nombres = entrenar_red_nombres_anatomicos()
        display(hist_red_nombres.tail())
    else:
        naming_model_final = cargar_red_nombres_anatomicos()
        hist_red_nombres = pd.DataFrame()
else:
    naming_model_final = None
    hist_red_nombres = pd.DataFrame()

## 13. Aplicar nombres anatomicos con restriccion monotonica

La red predice una probabilidad por caja, pero la columna tiene orden anatomico. Por eso se aplica una decodificacion monotonica: de arriba hacia abajo las etiquetas solo pueden avanzar de T1 a L5, y las cajas sobrantes pueden quedar como `extra`. Esto busca mejorar la metrica estricta sin castigar la visualizacion de vertebras adicionales en imagenes con GT parcial.


In [ ]:
def logits_nombres_para_prompts(prompts_auto, img_rgb, model_nombres):
    """Ejecuta la red de nombres sobre los prompts automaticos de un paciente."""
    rows = extraer_features_secuencia_cajas(prompts_auto, img_rgb)
    if not rows:
        return rows, np.zeros((0, NOMBRES_N_CLASSES), dtype=np.float32), np.zeros((0, NOMBRES_N_CLASSES), dtype=np.float32)
    x = np.array([[row[f] for f in model_nombres.features] for row in rows], dtype=np.float32)
    x = ((x - model_nombres.mean) / model_nombres.std).astype(np.float32)
    xt = torch.from_numpy(x[None, ...]).to(DEVICE)
    lengths = torch.tensor([len(rows)], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        logits = model_nombres(xt, lengths)[0, : len(rows)].detach().cpu().numpy()
    z = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(z) / (np.exp(z).sum(axis=1, keepdims=True) + 1e-9)
    return rows, logits, probs


def decodificar_nombres_monotonicos(logits):
    """Viterbi simple: permite extras y saltos anatomicos con penalizacion suave."""
    n = int(logits.shape[0])
    if n == 0:
        return []
    z = logits - logits.max(axis=1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(axis=1, keepdims=True) + 1e-9)

    # state = ultimo indice vertebral usado + 1. state=0 significa que aun no se asigno ninguna vertebra.
    dp = np.full((n + 1, N_CLASES + 1), -1e9, dtype=np.float32)
    prev = [[None for _ in range(N_CLASES + 1)] for _ in range(n + 1)]
    dp[0, 0] = 0.0

    for i in range(n):
        for state in range(N_CLASES + 1):
            base = dp[i, state]
            if base < -1e8:
                continue

            # Caja extra: se visualiza, pero no entra a metrica estricta.
            score_extra = base + float(logp[i, NOMBRES_EXTRA_ID]) - NOMBRES_EXTRA_PENALTY
            if score_extra > dp[i + 1, state]:
                dp[i + 1, state] = score_extra
                prev[i + 1][state] = (state, None)

            last_label = state - 1
            for label_idx in range(last_label + 1, N_CLASES):
                skip = label_idx - last_label - 1
                score = base + float(logp[i, label_idx]) - NOMBRES_SKIP_PENALTY * skip
                if score > dp[i + 1, label_idx + 1]:
                    dp[i + 1, label_idx + 1] = score
                    prev[i + 1][label_idx + 1] = (state, label_idx)

    state = int(np.argmax(dp[n]))
    labels = [None] * n
    for i in range(n, 0, -1):
        item = prev[i][state]
        if item is None:
            break
        prev_state, label_idx = item
        labels[i - 1] = label_idx
        state = prev_state
    return labels


def relabel_prompts_con_red_nombres(prompts_auto, img_rgb, model_nombres):
    """Renombra prompts automaticos con la red de nombres y la restriccion anatomica."""
    rows, logits, probs = logits_nombres_para_prompts(prompts_auto, img_rgb, model_nombres)
    if len(rows) == 0:
        return {}, pd.DataFrame()

    if NOMBRES_USAR_DP_MONOTONICA:
        decoded = decodificar_nombres_monotonicos(logits)
    else:
        decoded = [int(np.argmax(p[:N_CLASES])) if float(np.max(p[:N_CLASES])) >= NOMBRES_MIN_PROB_LABEL else None for p in probs]

    renombrados = {}
    asignaciones = []
    source_items = ordenar_prompts_vertical(prompts_auto)
    for row, (raw_label, info), label_idx in zip(rows, source_items, decoded):
        prob_extra = float(probs[row["rank"], NOMBRES_EXTRA_ID])
        if label_idx is None:
            asignaciones.append({
                "raw_label": raw_label,
                "vertebra_nn": NOMBRES_EXTRA_LABEL,
                "prob_nn": prob_extra,
                "prob_extra": prob_extra,
                "rank": int(row["rank"]),
                "estado_nombre": "extra",
            })
            continue

        prob_label = float(probs[row["rank"], label_idx])
        if prob_label < NOMBRES_MIN_PROB_LABEL and prob_extra > prob_label:
            asignaciones.append({
                "raw_label": raw_label,
                "vertebra_nn": NOMBRES_EXTRA_LABEL,
                "prob_nn": prob_label,
                "prob_extra": prob_extra,
                "rank": int(row["rank"]),
                "estado_nombre": "extra_por_baja_prob",
            })
            continue

        new_label = ID_TO_VERTEBRA[label_idx + 1]
        nuevo = dict(info)
        nuevo["vertebra_original"] = raw_label
        nuevo["id_real_original"] = VERTEBRA_TO_ID.get(raw_label, np.nan)
        nuevo["vertebra"] = new_label
        nuevo["id_real"] = label_idx + 1
        nuevo["nombre_nn_prob"] = prob_label
        nuevo["nombre_nn_extra_prob"] = prob_extra
        nuevo["nombre_nn_rank"] = int(row["rank"])
        nuevo["prompt_origen"] = "cajas_nn_red_nombres_anatomicos"
        if new_label not in renombrados or prob_label > renombrados[new_label].get("nombre_nn_prob", -1):
            renombrados[new_label] = nuevo

        asignaciones.append({
            "raw_label": raw_label,
            "vertebra_nn": new_label,
            "prob_nn": prob_label,
            "prob_extra": prob_extra,
            "rank": int(row["rank"]),
            "estado_nombre": "nombrada",
        })

    df_asig = pd.DataFrame(asignaciones)
    return renombrados, df_asig


def evaluar_red_nombres_split(split="val", patient_ids=None):
    """Compara cajas crudas, shift con GT y red de nombres en pacientes de validacion/test."""
    if not USAR_RED_NOMBRES_ANATOMICOS or globals().get("naming_model_final") is None:
        print("Red de nombres inactiva; no se evalua.")
        return pd.DataFrame()
    patient_ids = patient_ids or sorted(PROMPTS_DICC[split].keys())
    filas, detalles, asignaciones_all = [], [], []
    for patient_id in tqdm(patient_ids, desc=f"eval nombres {split}"):
        img, mask_gt, prompts_raw, _, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=ETIQUETADO_MODO)
        vertebras_gt = vertebras_gt_eval(mask_gt)
        prompts_nn, df_asig = relabel_prompts_con_red_nombres(prompts_raw, img, naming_model_final)
        best_shift, _, _ = evaluar_shifts_anatomicos(prompts_raw, mask_gt)
        prompts_shift, _ = relabel_prompts_por_shift(prompts_raw, best_shift)

        df_raw = evaluar_cajas(prompts_raw, mask_gt, vertebras_eval=vertebras_gt)
        df_shift = evaluar_cajas(prompts_shift, mask_gt, vertebras_eval=vertebras_gt)
        df_nn = evaluar_cajas(prompts_nn, mask_gt, vertebras_eval=vertebras_gt)
        for nombre, df_met in [("raw", df_raw), ("shift_gt", df_shift), ("red_nombres", df_nn)]:
            filas.append({
                "split": split,
                "patient_id": patient_id,
                "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
                "estrategia": nombre,
                "n_gt": len(vertebras_gt),
                "n_prompts_eval": len(prompts_nn) if nombre == "red_nombres" else (len(prompts_shift) if nombre == "shift_gt" else len(prompts_raw)),
                "best_shift": int(best_shift),
                "bbox_iou_estricto": float(df_met["bbox_iou"].mean()) if not df_met.empty else np.nan,
                "bbox_recall_estricto": float(df_met["bbox_recall"].mean()) if not df_met.empty else np.nan,
                "bbox_iou_flexible": float(df_met["bbox_iou_flexible"].mean()) if not df_met.empty else np.nan,
                "bbox_recall_flexible": float(df_met["bbox_recall_flexible"].mean()) if not df_met.empty else np.nan,
                "n_misma_etiqueta_flexible": int(df_met["flexible_misma_etiqueta"].sum()) if not df_met.empty and "flexible_misma_etiqueta" in df_met else 0,
            })
            if not df_met.empty:
                detalles.append(df_met.assign(split=split, patient_id=patient_id, estrategia=nombre))
        if not df_asig.empty:
            asignaciones_all.append(df_asig.assign(split=split, patient_id=patient_id))

    df_res = pd.DataFrame(filas)
    df_det = pd.concat(detalles, ignore_index=True) if detalles else pd.DataFrame()
    df_asig_all = pd.concat(asignaciones_all, ignore_index=True) if asignaciones_all else pd.DataFrame()
    df_res.to_csv(NN_SAM_DIR / f"red_nombres_{split}_resumen.csv", index=False)
    df_det.to_csv(NN_SAM_DIR / f"red_nombres_{split}_detalle.csv", index=False)
    df_asig_all.to_csv(NN_SAM_DIR / f"red_nombres_{split}_asignaciones.csv", index=False)
    display(df_res.groupby(["estrategia", "tipo_real"], as_index=False).agg(
        pacientes=("patient_id", "nunique"),
        bbox_iou_estricto=("bbox_iou_estricto", "mean"),
        bbox_recall_estricto=("bbox_recall_estricto", "mean"),
        bbox_iou_flexible=("bbox_iou_flexible", "mean"),
        bbox_recall_flexible=("bbox_recall_flexible", "mean"),
    ))
    return df_res


if USAR_RED_NOMBRES_ANATOMICOS:
    df_red_nombres_val = evaluar_red_nombres_split("val", PACIENTES_VISUALIZACION)
else:
    df_red_nombres_val = pd.DataFrame()

## 14. MedSAM final

En la version final solo queda la ruta ganadora:

- Prompt: `box_only`.
- Caja: `sin_pad`.
- Modelo final: `medsam_decoder_encoder_parcial`.

El decoder se entrena primero porque sirve como inicializacion para el ajuste final con encoder parcial.


## 15. Prompt final para MedSAM

`box_only` significa que MedSAM recibe solo la caja `[x0, y0, x1, y1]`. `sin_pad` significa que la caja se usa tal como la genero NN-SAM, sin expansion adicional.

Se eliminan los modos con puntos y las expansiones de caja porque no fueron la ruta ganadora.


In [ ]:
# Expansion opcional mantenida solo por compatibilidad; la configuracion final usa frac_x=0 y frac_y=0.
def expandir_bbox_xyxy_frac(bbox, img_shape, frac_x=0.0, frac_y=0.0):
    H, W = img_shape[:2]
    x0, y0, x1, y1 = [float(v) for v in bbox]
    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2
    bw = max(1.0, x1 - x0)
    bh = max(1.0, y1 - y0)
    dx = bw * float(frac_x)
    dy = bh * float(frac_y)
    return np.array([
        np.clip(x0 - dx, 0, W - 1),
        np.clip(y0 - dy, 0, H - 1),
        np.clip(x1 + dx, 0, W - 1),
        np.clip(y1 + dy, 0, H - 1),
    ], dtype=np.float32)


def preparar_prompt_medsam(img_rgb, bbox, modo="box_only", bbox_frac_x=0.0, bbox_frac_y=0.0):
    """Prompt final: solo caja, sin puntos positivos/negativos."""
    if modo != "box_only":
        raise ValueError("La version final SAM solo usa prompt_mode='box_only'.")
    bbox_np = expandir_bbox_xyxy_frac(bbox, img_rgb.shape, bbox_frac_x, bbox_frac_y)
    return {"box": bbox_np, "multimask_output": False}


def dice_iou_binario(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = int(np.logical_and(pred, gt).sum())
    union = int(np.logical_or(pred, gt).sum())
    denom = int(pred.sum() + gt.sum())
    return {
        "dice": (2 * inter / denom) if denom else np.nan,
        "iou": (inter / union) if union else np.nan,
        "pix_pred": int(pred.sum()),
        "pix_gt": int(gt.sum()),
    }


## 16. Entrenamiento MedSAM final

Se conserva solo el entrenamiento necesario para obtener el modelo final:

1. Entrenar `mask_decoder`.
2. Cargar ese checkpoint y entrenar `mask_decoder + ultimo bloque del encoder`.

El segundo checkpoint es el modelo final que se usa para validacion y test.


In [ ]:
from torch.utils.data import DataLoader


def collate_fn_medsam(batch):
    return batch


def extraer_state_dict_medsam_entrenamiento(state):
    """Acepta checkpoints guardados como state_dict directo o envueltos en claves comunes."""
    if isinstance(state, dict):
        for key in ["model", "model_state_dict", "state_dict", "medsam"]:
            if key in state and isinstance(state[key], dict):
                return state[key]
    return state


# Dataset por instancia vertebral: cada muestra es una imagen, una caja y la mascara binaria de una vertebra.
class MedSAMVertebraDatasetEntrega(Dataset):
    def __init__(self, split, frac_x=0.04, frac_y=0.06):
        self.split = split
        self.frac_x = frac_x
        self.frac_y = frac_y
        self.samples = []

        for patient_id, prompts_sample in PROMPTS_DICC[split].items():
            labels_gt = set(GT_LABELS_DICC.get(split, {}).get(patient_id, []))
            for vertebra, info in prompts_sample.items():
                if vertebra not in labels_gt:
                    continue
                self.samples.append({
                    "patient_id": patient_id,
                    "vertebra": vertebra,
                    "bbox": info["bbox_xyxy"],
                    "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
                })

        print(f"MedSAM {split}: {len(self.samples)} muestras vertebrales")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        path_img, path_mask = resolver_paths_muestra(self.split, item["patient_id"])
        img = cargar_imagen(path_img).astype(np.uint8)
        mask = cargar_mascara(path_mask)

        bbox = expandir_bbox_xyxy_frac(item["bbox"], img.shape, self.frac_x, self.frac_y)
        gt_bin = mask_binaria_vertebra(mask, item["vertebra"]).astype(np.float32)

        peso = MEDSAM_PESO_ESCOLIOSIS if item["tipo_real"] == "escoliosis" else MEDSAM_PESO_NORMAL
        if item["vertebra"] in MEDSAM_VERTEBRAS_DIFICILES:
            peso *= MEDSAM_PESO_VERTEBRA_DIFICIL

        return {
            "patient_id": item["patient_id"],
            "vertebra": item["vertebra"],
            "tipo_real": item["tipo_real"],
            "image": img,
            "box": torch.tensor(bbox, dtype=torch.float32),
            "gt_mask": torch.tensor(gt_bin[None, :, :], dtype=torch.float32),
            "sample_weight": torch.tensor(float(peso), dtype=torch.float32),
        }


# DataLoaders de MedSAM: batch pequeno por memoria GPU; collate conserva numpy images.
def crear_loaders_medsam_entrega():
    train_ds = MedSAMVertebraDatasetEntrega("train", frac_x=MEDSAM_TRAIN_FRAC_X, frac_y=MEDSAM_TRAIN_FRAC_Y)
    val_ds = MedSAMVertebraDatasetEntrega("val", frac_x=MEDSAM_TRAIN_FRAC_X, frac_y=MEDSAM_TRAIN_FRAC_Y)
    train_loader = DataLoader(
        train_ds,
        batch_size=MEDSAM_TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=MEDSAM_TRAIN_WORKERS,
        collate_fn=collate_fn_medsam,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=MEDSAM_TRAIN_BATCH_SIZE,
        shuffle=False,
        num_workers=MEDSAM_TRAIN_WORKERS,
        collate_fn=collate_fn_medsam,
    )
    return train_loader, val_loader


bce_loss_medsam = nn.BCEWithLogitsLoss()


def dice_loss_from_logits_medsam(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.reshape(probs.shape[0], -1)
    targets = targets.reshape(targets.shape[0], -1)
    inter = (probs * targets).sum(dim=1)
    denom = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()


@torch.no_grad()
def dice_iou_from_logits_medsam(logits, targets):
    pred = (torch.sigmoid(logits) > 0.5).float()
    pred_f = pred.reshape(pred.shape[0], -1)
    tgt_f = targets.reshape(targets.shape[0], -1)
    inter = (pred_f * tgt_f).sum(dim=1)
    union = ((pred_f + tgt_f) > 0).float().sum(dim=1)
    denom = pred_f.sum(dim=1) + tgt_f.sum(dim=1)
    dice = ((2 * inter) / denom.clamp(min=1)).mean().item()
    iou = (inter / union.clamp(min=1)).mean().item()
    return dice, iou


# Carga MedSAM base; desde aqui parten decoder y encoder parcial.
def cargar_medsam_base_para_entrenar():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA no esta disponible. No entreno MedSAM en CPU.")
    if not MEDSAM_CKPT_PATH.exists():
        raise FileNotFoundError(f"No existe MedSAM base: {MEDSAM_CKPT_PATH}")
    from segment_anything import sam_model_registry
    medsam = sam_model_registry["vit_b"](checkpoint=str(MEDSAM_CKPT_PATH)).to(DEVICE)
    return medsam


# Congela/libera parametros segun la variante a entrenar.
def configurar_entrenabilidad_medsam(medsam, modo):
    for p in medsam.parameters():
        p.requires_grad = False

    for p in medsam.prompt_encoder.parameters():
        p.requires_grad = False

    for p in medsam.mask_decoder.parameters():
        p.requires_grad = True

    if modo == "decoder_encoder_parcial":
        if not hasattr(medsam.image_encoder, "blocks"):
            raise AttributeError("No encontre image_encoder.blocks para abrir el ultimo bloque del encoder.")
        for p in medsam.image_encoder.blocks[-1].parameters():
            p.requires_grad = True

    n_total = sum(p.numel() for p in medsam.parameters())
    n_train = sum(p.numel() for p in medsam.parameters() if p.requires_grad)
    print(f"Modo {modo}: parametros entrenables {n_train:,} / {n_total:,} ({100*n_train/n_total:.2f}%)")


# Usa un solo LR para decoder y dos LR cuando se libera encoder parcial.
def optimizador_medsam(medsam, modo):
    if modo == "decoder":
        return torch.optim.AdamW(
            [p for p in medsam.mask_decoder.parameters() if p.requires_grad],
            lr=MEDSAM_LR_DECODER,
            weight_decay=MEDSAM_WEIGHT_DECAY,
        )

    params_encoder = []
    params_decoder = []
    for name, p in medsam.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("image_encoder.blocks."):
            params_encoder.append(p)
        elif name.startswith("mask_decoder."):
            params_decoder.append(p)

    return torch.optim.AdamW(
        [
            {"params": params_encoder, "lr": MEDSAM_LR_ENCODER},
            {"params": params_decoder, "lr": MEDSAM_LR_DECODER_REFINO},
        ],
        weight_decay=MEDSAM_WEIGHT_DECAY,
    )


# Forward diferenciable: necesario para que el ultimo bloque del encoder pueda aprender.
def forward_medsam_diferenciable(medsam, image_np, box_tensor, entrenar_encoder=False):
    H, W = image_np.shape[:2]
    image_t = torch.as_tensor(image_np, dtype=torch.float32, device=DEVICE).permute(2, 0, 1).unsqueeze(0)
    input_image = medsam.preprocess(image_t)

    if entrenar_encoder:
        image_embedding = medsam.image_encoder(input_image)
    else:
        with torch.no_grad():
            image_embedding = medsam.image_encoder(input_image)

    box_torch = box_tensor.to(DEVICE).float().reshape(1, 4)
    with torch.no_grad():
        sparse_embeddings, dense_embeddings = medsam.prompt_encoder(points=None, boxes=box_torch, masks=None)

    low_res_masks, iou_predictions = medsam.mask_decoder(
        image_embeddings=image_embedding,
        image_pe=medsam.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=False,
    )

    masks = medsam.postprocess_masks(low_res_masks, input_size=(H, W), original_size=(H, W))
    return masks, iou_predictions


# Ejecuta una epoca de entrenamiento o validacion y reporta loss, Dice e IoU binarios.
def run_epoch_medsam_entrega(medsam, loader, optimizer=None, modo="decoder", epoch=None):
    train = optimizer is not None
    medsam.train(train)
    # Prompt encoder queda congelado aunque el modelo este en modo train.
    medsam.prompt_encoder.eval()
    if modo == "decoder":
        medsam.image_encoder.eval()
    desc = f"MedSAM {modo} epoch {epoch}" if epoch is not None else f"MedSAM {modo}"

    total_loss = total_bce = total_dice_loss = total_dice = total_iou = 0.0
    n = 0
    entrenar_encoder = train and modo == "decoder_encoder_parcial"

    for batch in tqdm(loader, desc=desc, leave=True):
        for sample in batch:
            gt = sample["gt_mask"].to(DEVICE)
            weight = sample["sample_weight"].to(DEVICE)
            if gt.ndim == 3:
                gt = gt.unsqueeze(0)

            if train:
                optimizer.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                    logits, _ = forward_medsam_diferenciable(
                        medsam,
                        sample["image"],
                        sample["box"],
                        entrenar_encoder=entrenar_encoder,
                    )
                    loss_bce = bce_loss_medsam(logits, gt)
                    loss_dice = dice_loss_from_logits_medsam(logits, gt)
                    loss = loss_bce + loss_dice
                    if train:
                        loss = loss * weight

                if train:
                    loss.backward()
                    optimizer.step()

            dice_val, iou_val = dice_iou_from_logits_medsam(logits.detach(), gt)
            total_loss += float(loss.detach().cpu())
            total_bce += float(loss_bce.detach().cpu())
            total_dice_loss += float(loss_dice.detach().cpu())
            total_dice += dice_val
            total_iou += iou_val
            n += 1

    return {
        "loss": total_loss / max(n, 1),
        "bce": total_bce / max(n, 1),
        "dice_loss": total_dice_loss / max(n, 1),
        "dice_bin": total_dice / max(n, 1),
        "iou_bin": total_iou / max(n, 1),
    }


# Entrena una variante, guarda el mejor checkpoint y exporta historial CSV.
def entrenar_variante_medsam_entrega(
    modo,
    train_loader,
    val_loader,
    epochs,
    paciencia,
    out_path,
    init_state_path=None,
):
    medsam = cargar_medsam_base_para_entrenar()
    if init_state_path is not None and Path(init_state_path).exists():
        state = torch.load(init_state_path, map_location=DEVICE)
        state = extraer_state_dict_medsam_entrenamiento(state)
        missing, unexpected = medsam.load_state_dict(state, strict=False)
        print(f"Inicializado desde {init_state_path} | faltantes={len(missing)} | inesperadas={len(unexpected)}")

    configurar_entrenabilidad_medsam(medsam, modo)
    optimizer = optimizador_medsam(medsam, modo)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    best_val = float("inf")
    best_state = None
    sin_mejora = 0
    hist = []

    for epoch in range(1, int(epochs) + 1):
        train_metrics = run_epoch_medsam_entrega(medsam, train_loader, optimizer=optimizer, modo=modo, epoch=epoch)
        val_metrics = run_epoch_medsam_entrega(medsam, val_loader, optimizer=None, modo=modo, epoch=epoch)
        scheduler.step(val_metrics["loss"])

        row = {
            "modo": modo,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_bce": train_metrics["bce"],
            "train_dice_loss": train_metrics["dice_loss"],
            "train_dice_bin": train_metrics["dice_bin"],
            "train_iou_bin": train_metrics["iou_bin"],
            "val_loss": val_metrics["loss"],
            "val_bce": val_metrics["bce"],
            "val_dice_loss": val_metrics["dice_loss"],
            "val_dice_bin": val_metrics["dice_bin"],
            "val_iou_bin": val_metrics["iou_bin"],
            "lr_min": min(g["lr"] for g in optimizer.param_groups),
            "lr_max": max(g["lr"] for g in optimizer.param_groups),
        }
        hist.append(row)
        print(
            f"{modo} epoch={epoch:02d} train_loss={row['train_loss']:.4f} "
            f"val_loss={row['val_loss']:.4f} val_dice={row['val_dice_bin']:.4f} val_iou={row['val_iou_bin']:.4f}"
        )

        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in medsam.state_dict().items()}
            sin_mejora = 0
            print("  -> nuevo mejor checkpoint")
        else:
            sin_mejora += 1
            print(f"  -> sin mejora ({sin_mejora}/{paciencia})")
            if sin_mejora >= paciencia:
                print("Early stopping MedSAM")
                break

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in medsam.state_dict().items()}

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model": best_state,
        "modo": modo,
        "config": {
            "frac_x": MEDSAM_TRAIN_FRAC_X,
            "frac_y": MEDSAM_TRAIN_FRAC_Y,
            "epochs_max": epochs,
            "paciencia": paciencia,
        },
    }, out_path)
    hist_df = pd.DataFrame(hist)
    hist_df.to_csv(RESULTADOS_DIR / f"historial_medsam_{modo}.csv", index=False)
    del medsam
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Checkpoint guardado:", out_path)
    return hist_df


# Secuencia de entrega: primero decoder, luego encoder parcial inicializado desde decoder.
if EJECUTAR_ENTRENAMIENTO_MEDSAM:
    train_loader_medsam, val_loader_medsam = crear_loaders_medsam_entrega()
    hist_medsam_decoder = entrenar_variante_medsam_entrega(
        modo="decoder",
        train_loader=train_loader_medsam,
        val_loader=val_loader_medsam,
        epochs=MEDSAM_DECODER_EPOCHS,
        paciencia=MEDSAM_DECODER_PATIENCIA,
        out_path=MEDSAM_DECODER_PATH,
    )
    display(hist_medsam_decoder.tail())

    hist_medsam_encoder_parcial = entrenar_variante_medsam_entrega(
        modo="decoder_encoder_parcial",
        train_loader=train_loader_medsam,
        val_loader=val_loader_medsam,
        epochs=MEDSAM_ENCODER_PARCIAL_EPOCHS,
        paciencia=MEDSAM_ENCODER_PARCIAL_PATIENCIA,
        out_path=MEDSAM_ENCODER_DECODER_PATH,
        init_state_path=MEDSAM_DECODER_PATH,
    )
    display(hist_medsam_encoder_parcial.tail())
else:
    print("Entrenamiento MedSAM desactivado. Se usaran checkpoints existentes si estan en las rutas configuradas.")


## 17. Cargar el modelo final para evaluacion

La lista de variantes queda reducida a una sola: `medsam_decoder_encoder_parcial`. El baseline general y el decoder solo ya no se evaluan en esta version final.


In [ ]:
# Permite leer checkpoints guardados con distintas convenciones.
def extraer_state_dict_medsam(state):
    """Acepta checkpoints guardados como state_dict directo o envueltos en claves comunes."""
    if isinstance(state, dict):
        for key in ["model", "model_state_dict", "state_dict", "medsam"]:
            if key in state and isinstance(state[key], dict):
                return state[key]
    return state


# Carga una variante entrenada y la envuelve en SamPredictor para inferencia rapida.
def cargar_predictor_medsam_variante(variante):
    """Carga MedSAM base y, opcionalmente, pesos ya entrenados."""
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA no esta disponible. No se ejecuta MedSAM en CPU para evitar tiempos largos.")
    if not MEDSAM_CKPT_PATH.exists():
        raise FileNotFoundError(f"No existe el checkpoint base MedSAM: {MEDSAM_CKPT_PATH}")

    from segment_anything import sam_model_registry, SamPredictor

    device_medsam = torch.device("cuda")
    torch.cuda.empty_cache()

    model_medsam = sam_model_registry["vit_b"](checkpoint=str(MEDSAM_CKPT_PATH)).to(device_medsam)
    state_path = variante.get("state_path")
    carga_info = {
        "variante": variante["nombre"],
        "descripcion": variante["descripcion"],
        "state_path": str(state_path) if state_path else "",
        "missing": np.nan,
        "unexpected": np.nan,
    }

    if state_path:
        state_path = Path(state_path)
        if not state_path.exists():
            raise FileNotFoundError(f"No existe checkpoint de variante {variante['nombre']}: {state_path}")
        state = torch.load(state_path, map_location=device_medsam)
        state = extraer_state_dict_medsam(state)
        missing, unexpected = model_medsam.load_state_dict(state, strict=False)
        carga_info["missing"] = len(missing)
        carga_info["unexpected"] = len(unexpected)
        print(f"{variante['nombre']} cargado:", state_path, "| faltantes:", len(missing), "| inesperadas:", len(unexpected))
    else:
        print(f"{variante['nombre']} cargado desde MedSAM base:", MEDSAM_CKPT_PATH)

    model_medsam.eval()
    return SamPredictor(model_medsam), carga_info


# Lista final de modelos que entra al comparativo con las mismas cajas NN-SAM.
MEDSAM_VARIANTES_NN_SAM = [
    {
        "nombre": "medsam_decoder_encoder_parcial",
        "descripcion": "Modelo final: MedSAM con decoder y ultimo bloque del encoder ajustados",
        "state_path": MEDSAM_ENCODER_DECODER_PATH,
    },
]


## 18. Cajas automaticas usadas por el MedSAM final

MedSAM final recibe todas las cajas detectadas por NN-SAM. La metrica estricta puede aplicar shift para evaluar etiquetas; la metrica flexible permite saber si la vertebra fue segmentada aunque el nombre este corrido.


In [ ]:
# Genera prompts automaticos y, opcionalmente, aplica shift solo para evaluacion estricta.
def obtener_prompts_nn_sam_para_medsam(split, patient_id, usar_shift=True):
    """Devuelve prompts crudos y prompts renombrados para metrica estricta.

    Prioridad final:
    1. Red neuronal de nombres anatomicos, si esta activa y entrenada.
    2. Shift anatomico como respaldo diagnostico.
    3. Prompts crudos si no hay ninguna correccion disponible.
    """
    img, mask_gt, prompts_raw, _, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=ETIQUETADO_MODO)
    best_shift = 0
    prompts_eval = prompts_raw

    if USAR_RED_NOMBRES_ANATOMICOS and globals().get("naming_model_final") is not None:
        prompts_named, _ = relabel_prompts_con_red_nombres(prompts_raw, img, globals().get("naming_model_final"))
        if len(prompts_named) >= NOMBRES_MIN_NOMBRADAS_FALLBACK:
            return img, mask_gt, prompts_raw, prompts_named, best_shift

    if usar_shift:
        best_shift, _, _ = evaluar_shifts_anatomicos(prompts_raw, mask_gt)
        prompts_eval, _ = relabel_prompts_por_shift(prompts_raw, best_shift)
    return img, mask_gt, prompts_raw, prompts_eval, best_shift


def centro_y_bbox(info):
    x0, y0, x1, y1 = [float(v) for v in info["bbox_xyxy"]]
    return (y0 + y1) / 2


# Etiquetado visual top-down/bottom-up para inspeccionar imagenes con GT parcial.
def asignar_labels_por_orden(prompts_raw, modo="top_down"):
    """Asigna etiquetas anatomicas solo para visualizacion segun orden vertical."""
    items = sorted(prompts_raw.items(), key=lambda kv: centro_y_bbox(kv[1]))
    n = min(len(items), N_CLASES)
    if modo == "bottom_up":
        labels = CLASES_OBJETIVO[N_CLASES - n:]
    else:
        labels = CLASES_OBJETIVO[:n]
    return {raw_label: labels[i] for i, (raw_label, _) in enumerate(items[:n])}


def construir_mascara_semantica_desde_preds(pred_masks, labels_forzados=None):
    """Construye una mascara semantica desde mascaras binarias por vertebra."""
    if not pred_masks:
        return None
    first = next(iter(pred_masks.values()))
    mask_sem = np.zeros(first.shape, dtype=np.uint8)
    for vertebra, pred in sorted(pred_masks.items(), key=lambda kv: VERTEBRA_TO_ID.get(kv[0], 999)):
        label = labels_forzados.get(vertebra, vertebra) if labels_forzados else vertebra
        id_v = VERTEBRA_TO_ID.get(label, 0)
        if id_v > 0:
            mask_sem[pred.astype(bool)] = id_v
    return mask_sem


# Metrica flexible: para cada vertebra GT busca la mejor mascara segmentada.
def evaluar_predicciones_flexibles(pred_masks, mask_gt, vertebras_gt):
    """Para cada vertebra GT busca la mejor mascara predicha entre todas las cajas segmentadas."""
    filas = []
    for gt_v in vertebras_gt:
        gt = mask_binaria_vertebra(mask_gt, gt_v).astype(bool)
        mejor = None
        mejor_key = None
        for pred_v, pred in pred_masks.items():
            met = dice_iou_binario(pred, gt)
            key = (met["dice"], met["iou"])
            if mejor is None or key > mejor_key:
                pred_id = VERTEBRA_TO_ID.get(pred_v, np.nan)
                gt_id = VERTEBRA_TO_ID.get(gt_v, np.nan)
                mejor = {
                    "gt_vertebra": gt_v,
                    "gt_id": gt_id,
                    "prompt_flexible_vertebra": pred_v,
                    "prompt_flexible_id": pred_id,
                    "dice_flexible": float(met["dice"]) if np.isfinite(met["dice"]) else np.nan,
                    "iou_flexible": float(met["iou"]) if np.isfinite(met["iou"]) else np.nan,
                    "pix_gt": int(met["pix_gt"]),
                    "pix_pred_flexible": int(met["pix_pred"]),
                    "flexible_misma_etiqueta": bool(pred_v == gt_v),
                    "desfase_id_flexible": int(pred_id - gt_id) if np.isfinite(pred_id) and np.isfinite(gt_id) else np.nan,
                }
                mejor_key = key
        if mejor is not None:
            filas.append(mejor)
    return pd.DataFrame(filas)


# Segmenta un paciente completo con una variante MedSAM y guarda tablas/figuras.
def segmentar_paciente_con_predictor(
    predictor,
    variante_nombre,
    patient_id="S_187",
    split="val",
    usar_shift=True,
    prompt_mode="box_only",
    max_vertebras=None,
    guardar=True,
    bbox_expansion_nombre="sin_pad",
    bbox_frac_x=0.0,
    bbox_frac_y=0.0,
):
    img, mask_gt, prompts_raw, prompts_eval, best_shift = obtener_prompts_nn_sam_para_medsam(split, patient_id, usar_shift=usar_shift)
    predictor.set_image(img)

    vertebras_gt = vertebras_gt_eval(mask_gt)
    labels_gt_set = set(vertebras_gt)
    pred_masks = {}
    detalles = []

    labels_top_down = asignar_labels_por_orden(prompts_raw, modo="top_down")
    labels_bottom_up = asignar_labels_por_orden(prompts_raw, modo="bottom_up")

    # Relaciona cada caja cruda con la etiqueta corregida por shift usada para metrica estricta.
    raw_to_eval = {}
    for eval_label, info in prompts_eval.items():
        raw_label = info.get("vertebra_original", eval_label)
        raw_to_eval[raw_label] = eval_label

    # Se segmentan todas las cajas originales para visualizar toda la columna propuesta.
    items = sorted(prompts_raw.items(), key=lambda kv: centro_y_bbox(kv[1]))
    if max_vertebras is not None:
        items = items[:int(max_vertebras)]

    t0 = time.perf_counter()
    for raw_label, info in tqdm(items, desc=f"{variante_nombre} {patient_id}", leave=False):
        bbox = [int(v) for v in info["bbox_xyxy"]]
        bbox_prompt = expandir_bbox_xyxy_frac(bbox, img.shape, bbox_frac_x, bbox_frac_y)
        kwargs = preparar_prompt_medsam(img, bbox, modo=prompt_mode, bbox_frac_x=bbox_frac_x, bbox_frac_y=bbox_frac_y)
        masks, scores, logits = predictor.predict(**kwargs)
        pred = masks[0].astype(bool)
        pred_masks[raw_label] = pred

        eval_label = raw_to_eval.get(raw_label, raw_label)
        if eval_label in labels_gt_set:
            gt = mask_binaria_vertebra(mask_gt, eval_label).astype(bool)
            met = dice_iou_binario(pred, gt)
            estado_eval = "evaluado_gt"
        else:
            met = {"dice": np.nan, "iou": np.nan, "pix_pred": int(pred.sum()), "pix_gt": 0}
            estado_eval = "prompt_extra_sin_gt"

        detalles.append({
            "variante": variante_nombre,
            "patient_id": patient_id,
            "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
            "split": split,
            "vertebra": eval_label,
            "vertebra_original": raw_label,
            "visual_label_top_down": labels_top_down.get(raw_label, ""),
            "visual_label_bottom_up": labels_bottom_up.get(raw_label, ""),
            "shift_anatomico": int(best_shift),
            "prompt_mode": prompt_mode,
            "bbox_expansion": bbox_expansion_nombre,
            "bbox_frac_x": float(bbox_frac_x),
            "bbox_frac_y": float(bbox_frac_y),
            "estado_eval": estado_eval,
            "score_medsam": float(scores[0]),
            "dice_estricto": float(met["dice"]) if np.isfinite(met["dice"]) else np.nan,
            "iou_estricto": float(met["iou"]) if np.isfinite(met["iou"]) else np.nan,
            "pix_gt": int(met["pix_gt"]),
            "pix_pred": int(met["pix_pred"]),
            "bbox_xyxy": bbox,
            "bbox_prompt_xyxy": [float(v) for v in bbox_prompt],
        })

    tiempo_total = time.perf_counter() - t0
    df_prompts = pd.DataFrame(detalles)
    df_prompts["n_gt"] = len(vertebras_gt)
    df_prompts["n_segmentadas_total"] = len(items)
    df_prompts["tiempo_total_s"] = tiempo_total
    df_prompts["tiempo_por_vertebra_s"] = tiempo_total / max(len(df_prompts), 1)

    df_flexible = evaluar_predicciones_flexibles(pred_masks, mask_gt, vertebras_gt)
    if not df_flexible.empty:
        df_flexible["variante"] = variante_nombre
        df_flexible["patient_id"] = patient_id
        df_flexible["tipo_real"] = "escoliosis" if str(patient_id).startswith("S_") else "normal"
        df_flexible["split"] = split
        df_flexible["prompt_mode"] = prompt_mode
        df_flexible["bbox_expansion"] = bbox_expansion_nombre
        df_flexible["bbox_frac_x"] = float(bbox_frac_x)
        df_flexible["bbox_frac_y"] = float(bbox_frac_y)
        df_flexible["shift_anatomico"] = int(best_shift)
        df_flexible["n_gt"] = len(vertebras_gt)
        df_flexible["n_segmentadas_total"] = len(items)

    mascara_sem_original = construir_mascara_semantica_desde_preds(pred_masks)
    mascara_sem_top_down = construir_mascara_semantica_desde_preds(pred_masks, labels_forzados=labels_top_down)
    mascara_sem_bottom_up = construir_mascara_semantica_desde_preds(pred_masks, labels_forzados=labels_bottom_up)

    labels_flex = {}
    if not df_flexible.empty:
        for _, row in df_flexible.iterrows():
            labels_flex[row["prompt_flexible_vertebra"]] = row["gt_vertebra"]
    pred_masks_flex = {
        row["prompt_flexible_vertebra"]: pred_masks[row["prompt_flexible_vertebra"]]
        for _, row in df_flexible.iterrows()
        if row["prompt_flexible_vertebra"] in pred_masks
    } if not df_flexible.empty else {}
    mascara_sem_flexible = construir_mascara_semantica_desde_preds(pred_masks_flex, labels_forzados=labels_flex)

    if guardar and MEDSAM_GUARDAR_FIGURAS:
        out_base = f"medsam_nn_sam_{split}_{patient_id}_{variante_nombre}_{prompt_mode}_{bbox_expansion_nombre}"
        df_prompts.to_csv(NN_SAM_DIR / f"{out_base}_prompts.csv", index=False)
        df_flexible.to_csv(NN_SAM_DIR / f"{out_base}_flexible.csv", index=False)

        fig, ax = plt.subplots(1, 6, figsize=(34, 7))
        ax[0].imshow(img)
        ax[0].set_title("Radiografia")
        ax[1].imshow(mask_gt, cmap="nipy_spectral")
        ax[1].set_title(f"GT disponible | n={len(vertebras_gt)}")
        ax[2].imshow(mascara_sem_original, cmap="nipy_spectral")
        ax[2].set_title(f"Original/shift | n={len(items)}")
        ax[3].imshow(mascara_sem_top_down, cmap="nipy_spectral")
        ax[3].set_title("Visual top-down")
        ax[4].imshow(mascara_sem_bottom_up, cmap="nipy_spectral")
        ax[4].set_title("Visual bottom-up")
        ax[5].imshow(mascara_sem_flexible if mascara_sem_flexible is not None else np.zeros_like(mask_gt), cmap="nipy_spectral")
        ax[5].set_title("Mejor mascara flexible vs GT")
        for a in ax:
            a.axis("off")
        plt.tight_layout()
        plt.savefig(NN_SAM_DIR / f"{out_base}.png", dpi=140, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    return df_prompts, df_flexible


## 19. Evaluacion de validacion con la estrategia ganadora

Esta evaluacion usa solo:

- `medsam_decoder_encoder_parcial`
- `box_only`
- `sin_pad`

Ya no se comparan pads ni variantes descartadas. Esta celda sirve para confirmar comportamiento en el panel de validacion antes de pasar a `test`.


In [ ]:
# Ejecuta la evaluacion final: modelo ganador x sin_pad x pacientes trazadores.
def comparar_variantes_medsam_nn_sam(
    patient_ids=None,
    split="val",
    variantes=None,
    usar_shift=True,
    prompt_mode="box_only",
    max_vertebras=None,
    expansiones_bbox=None,
):
    patient_ids = patient_ids or MEDSAM_COMPARAR_PATIENT_IDS
    variantes = variantes or MEDSAM_VARIANTES_NN_SAM
    expansiones_bbox = expansiones_bbox or MEDSAM_EXPANSIONES_BBOX

    detalles_prompts = []
    detalles_flexibles = []
    cargas = []
    for variante in variantes:
        predictor, carga_info = cargar_predictor_medsam_variante(variante)
        cargas.append(carga_info)
        for expansion in expansiones_bbox:
            for pid in patient_ids:
                df_prompts_pid, df_flex_pid = segmentar_paciente_con_predictor(
                    predictor=predictor,
                    variante_nombre=variante["nombre"],
                    patient_id=pid,
                    split=split,
                    usar_shift=usar_shift,
                    prompt_mode=prompt_mode,
                    max_vertebras=max_vertebras,
                    guardar=True,
                    bbox_expansion_nombre=expansion["nombre"],
                    bbox_frac_x=expansion["frac_x"],
                    bbox_frac_y=expansion["frac_y"],
                )
                detalles_prompts.append(df_prompts_pid)
                detalles_flexibles.append(df_flex_pid)
        del predictor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df_prompts = pd.concat(detalles_prompts, ignore_index=True) if detalles_prompts else pd.DataFrame()
    df_flexible = pd.concat(detalles_flexibles, ignore_index=True) if detalles_flexibles else pd.DataFrame()
    df_cargas = pd.DataFrame(cargas)

    if df_prompts.empty:
        return pd.DataFrame(), pd.DataFrame(), df_prompts, df_flexible, df_cargas

    # Solo las mascaras con GT disponible entran a la metrica estricta.
    df_eval = df_prompts[df_prompts["estado_eval"] == "evaluado_gt"].copy()

    df_resumen_paciente = (
        df_eval
        .groupby(["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"], as_index=False)
        .agg(
            n_gt=("n_gt", "max"),
            n_eval_estricto=("vertebra", "count"),
            dice_estricto_promedio=("dice_estricto", "mean"),
            iou_estricto_promedio=("iou_estricto", "mean"),
            score_medsam_promedio=("score_medsam", "mean"),
            tiempo_total_s=("tiempo_total_s", "max"),
            tiempo_por_vertebra_s=("tiempo_por_vertebra_s", "mean"),
            shift_anatomico=("shift_anatomico", "max"),
        )
    )

    seg_counts = (
        df_prompts
        .groupby(["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"], as_index=False)
        .agg(n_segmentadas_total=("vertebra", "count"))
    )
    df_resumen_paciente = df_resumen_paciente.merge(
        seg_counts,
        on=["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"],
        how="left",
    )

    if not df_flexible.empty:
        flex_res = (
            df_flexible
            .groupby(["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"], as_index=False)
            .agg(
                n_eval_flexible=("gt_vertebra", "count"),
                dice_flexible_promedio=("dice_flexible", "mean"),
                iou_flexible_promedio=("iou_flexible", "mean"),
                n_flexible_misma_etiqueta=("flexible_misma_etiqueta", "sum"),
                desfase_flexible_promedio=("desfase_id_flexible", "mean"),
            )
        )
        df_resumen_paciente = df_resumen_paciente.merge(
            flex_res,
            on=["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"],
            how="left",
        )

    # Resumen por grupo clinico: normal vs escoliosis.
    df_resumen_tipo = (
        df_resumen_paciente
        .groupby(["variante", "tipo_real", "prompt_mode", "bbox_expansion"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            n_eval_estricto=("n_eval_estricto", "sum"),
            n_segmentadas_total=("n_segmentadas_total", "sum"),
            dice_estricto_promedio=("dice_estricto_promedio", "mean"),
            iou_estricto_promedio=("iou_estricto_promedio", "mean"),
            dice_flexible_promedio=("dice_flexible_promedio", "mean"),
            iou_flexible_promedio=("iou_flexible_promedio", "mean"),
            tiempo_por_vertebra_s=("tiempo_por_vertebra_s", "mean"),
        )
    )

    # Resumen global para escoger variante y expansion candidata.
    df_global = (
        df_resumen_paciente
        .groupby(["variante", "prompt_mode", "bbox_expansion"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            n_eval_estricto=("n_eval_estricto", "sum"),
            n_segmentadas_total=("n_segmentadas_total", "sum"),
            dice_estricto_promedio=("dice_estricto_promedio", "mean"),
            iou_estricto_promedio=("iou_estricto_promedio", "mean"),
            dice_flexible_promedio=("dice_flexible_promedio", "mean"),
            iou_flexible_promedio=("iou_flexible_promedio", "mean"),
            tiempo_por_vertebra_s=("tiempo_por_vertebra_s", "mean"),
        )
        .sort_values(["dice_flexible_promedio", "dice_estricto_promedio"], ascending=False)
    )

    df_prompts.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_detalle_prompts.csv", index=False)
    df_flexible.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_detalle_flexible.csv", index=False)
    df_resumen_paciente.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_resumen_paciente.csv", index=False)
    df_resumen_tipo.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_resumen_tipo.csv", index=False)
    df_global.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_resumen_global.csv", index=False)
    df_cargas.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_cargas.csv", index=False)

    display(df_global)
    display(df_resumen_tipo)
    display(df_resumen_paciente)
    display(df_cargas)
    return df_global, df_resumen_tipo, df_prompts, df_flexible, df_cargas


# Lanza la evaluacion final despues de entrenar/cargar las variantes.
if EJECUTAR_COMPARATIVO_MEDSAM:
    df_medsam_variantes_global, df_medsam_variantes_tipo, df_medsam_variantes_prompts, df_medsam_variantes_flexible, df_medsam_variantes_cargas = comparar_variantes_medsam_nn_sam(
        patient_ids=MEDSAM_COMPARAR_PATIENT_IDS,
        split=MEDSAM_DEMO_SPLIT,
        usar_shift=MEDSAM_COMPARAR_USAR_SHIFT,
        prompt_mode=MEDSAM_COMPARAR_PROMPT_MODE,
        max_vertebras=MEDSAM_DEMO_MAX_VERTEBRAS,
        expansiones_bbox=MEDSAM_EXPANSIONES_BBOX,
    )
elif EJECUTAR_MEDSAM_BASICO:
    df_medsam_demo = comparar_variantes_medsam_nn_sam(
        patient_ids=[MEDSAM_DEMO_PATIENT_ID],
        split=MEDSAM_DEMO_SPLIT,
        variantes=[MEDSAM_VARIANTES_NN_SAM[1]],
        usar_shift=MEDSAM_DEMO_USAR_SHIFT,
        prompt_mode=MEDSAM_COMPARAR_PROMPT_MODE,
        max_vertebras=MEDSAM_DEMO_MAX_VERTEBRAS,
        expansiones_bbox=MEDSAM_EXPANSIONES_BBOX,
    )[2]
else:
    print("MedSAM no se ejecuto. Activa EJECUTAR_COMPARATIVO_MEDSAM=True para comparar variantes.")


## 20. Test final

Despues de confirmar la ruta ganadora en validacion, se evalua la misma estrategia en `test`. Esta prueba no reentrena nada y no compara alternativas descartadas.


In [ ]:
# ============================================================
# TEST FINAL CON LA MEJOR CONFIGURACION EN VAL
# ============================================================
# Objetivo:
# Evaluar en test la ruta que mejor funciono en validacion:
#   - MedSAM decoder + encoder parcial
#   - prompts automaticos NN-SAM
#   - box_only
#   - sin expansion adicional de caja
#
# Esta celda NO reentrena. Solo carga el checkpoint ya entrenado
# y calcula metricas/figuras sobre test.

EJECUTAR_MEDSAM_TEST_FINAL = True
MEDSAM_TEST_USAR_SHIFT = True
MEDSAM_TEST_PROMPT_MODE = "box_only"
MEDSAM_TEST_MAX_VERTEBRAS = None
MEDSAM_TEST_GUARDAR_FIGURAS = True

# Usar todo el split test. Si quieres una prueba rapida primero,
# reemplaza esta lista por algunos IDs concretos.
MEDSAM_TEST_PATIENT_IDS = sorted(PROMPTS_DICC["test"].keys())

# Mejor configuracion segun val.
MEDSAM_TEST_VARIANTES = [
    {
        "nombre": "medsam_decoder_encoder_parcial",
        "descripcion": "MedSAM con decoder y ultimo bloque del encoder ajustados en este notebook",
        "state_path": MEDSAM_ENCODER_DECODER_PATH,
    }
]

MEDSAM_TEST_EXPANSIONES = [
    {"nombre": "sin_pad", "frac_x": 0.00, "frac_y": 0.00},
]

if EJECUTAR_MEDSAM_TEST_FINAL:
    # Reutiliza las funciones ya definidas en el notebook.
    # No vuelve a entrenar cajas ni MedSAM.
    df_medsam_test_global, df_medsam_test_tipo, df_medsam_test_prompts, df_medsam_test_flexible, df_medsam_test_cargas = comparar_variantes_medsam_nn_sam(
        patient_ids=MEDSAM_TEST_PATIENT_IDS,
        split="test",
        variantes=MEDSAM_TEST_VARIANTES,
        usar_shift=MEDSAM_TEST_USAR_SHIFT,
        prompt_mode=MEDSAM_TEST_PROMPT_MODE,
        max_vertebras=MEDSAM_TEST_MAX_VERTEBRAS,
        expansiones_bbox=MEDSAM_TEST_EXPANSIONES,
    )

    # Guardar con nombres explicitos para no confundir con val.
    df_medsam_test_global.to_csv(NN_SAM_DIR / "FINAL_TEST_medsam_nn_sam_resumen_global.csv", index=False)
    df_medsam_test_tipo.to_csv(NN_SAM_DIR / "FINAL_TEST_medsam_nn_sam_resumen_tipo.csv", index=False)
    df_medsam_test_prompts.to_csv(NN_SAM_DIR / "FINAL_TEST_medsam_nn_sam_detalle_prompts.csv", index=False)
    df_medsam_test_flexible.to_csv(NN_SAM_DIR / "FINAL_TEST_medsam_nn_sam_detalle_flexible.csv", index=False)

    display(df_medsam_test_global)
    display(df_medsam_test_tipo)


## 21. Conclusion y uso de esta version

Esta version queda como base para presentar y optimizar el mejor modelo:

`NN-SAM cajas automaticas + red de nombres anatomicos + MedSAM decoder_encoder_parcial + box_only + sin_pad`

La nueva red de nombres intenta cerrar la brecha entre metrica flexible y estricta. Si mejora el estricto sin bajar mucho el flexible, el siguiente paso sera consolidarla como parte permanente del pipeline final.

Las mejoras futuras deberian concentrarse en:

- revisar casos de escoliosis con desfase grande;
- fortalecer la red de nombres con features/crops si hace falta;
- mejorar visualizacion y reporte final;
- conservar el pipeline rapido para que MedSAM funcione como paso de segmentacion posterior a las cajas.
